# symbolics_prep.py

In [ ]:
%%writefile /kaggle/working/symbolics_integration.py
# -*- coding: utf-8 -*-
"""
Symbolics integration — Kaggle/JAX-safe (no CLI, no Python branching in jitted paths).
- Provides:
    * DEFAULT_PROTEIN_CONFIG
    * SymbolicsHelper(...).build_for_subset(seqs, max_len)
    * ensure_symbolic_params(params, num_symbols, rng_key)
    * make_train_step_symbolic(optimizer)
    * make_train_step_symbolic_pmap(optimizer, axis_name="dp")   <-- multi-GPU
    * get_pairs_for_batch(batch_indices, neighbors_local, ...)
- Includes encoder/decoder implementations that match the trainer.
"""

from __future__ import annotations
import re
from typing import List, Dict, Tuple, Optional

import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
import optax

# -------------------------- Minimal defaults ---------------------------
DEFAULT_PROTEIN_CONFIG = {
    "alphabet": "protein",
    "k_neighbors": 8,
    "symbols": [
        {"id": "P_LOOP_WA",    "regex": r"G....GKT[ST]"},
        {"id": "HExH_METALLO", "regex": r"H..H"},
        {"id": "CYS_KNOT",     "regex": r"C.{2}C.{4,8}C"},
    ],
}

# ---------------------- Dropout + Encode/Decode (mirror trainer) --------------------
def apply_dropout(x, key, rate):
    rate = jnp.asarray(rate, dtype=x.dtype)
    keep_prob = jnp.clip(1.0 - rate, 0.0, 1.0)
    def do_keep(_):
        mask = jax.random.bernoulli(key, p=keep_prob, shape=x.shape)
        return jnp.where(mask, x / jnp.maximum(keep_prob, 1e-6), 0.0)
    return lax.cond(jnp.equal(rate, 0.0), lambda _: x, do_keep, operand=None)

def _encode_step(carry, token_and_pos):
    state, mps, rng_key, rate = carry
    tok, pos = token_and_pos
    W = mps[pos, tok]
    state = state @ W
    key = jax.random.fold_in(rng_key, pos)
    state = apply_dropout(state, key, rate)
    return (state, mps, rng_key, rate), state

@jax.jit
def encode(enc_params, seq_indices, rng_key, dropout_rate):
    L = enc_params['mps'].shape[0]
    bond_dim_mps = enc_params['mps'].shape[-1]
    state0 = jnp.zeros((bond_dim_mps,), dtype=jnp.float32).at[0].set(1.0)
    positions = jnp.arange(L, dtype=jnp.int32)
    (_, _, _, _), states = lax.scan(
        _encode_step,
        (state0, enc_params['mps'], rng_key, dropout_rate),
        (seq_indices, positions)
    )
    # MERA pairwise reduction
    def pair_reduce(_, i):
        l = states[2*i]
        r = states[2*i + 1]
        iso = enc_params['isos'][i]
        hi = jnp.einsum("i,j,ijk->k", l, r, iso)
        return None, hi
    _, hi_list = lax.scan(pair_reduce, None, jnp.arange(L//2))
    tv = hi_list.reshape(-1)
    return tv / (jnp.linalg.norm(tv) + 1e-9)

def _decode_step(carry, tok_and_pos):
    vec, ctx, mps_dec, out_proj, rng_key, rate, first_flag = carry
    tok, pos = tok_and_pos
    W = mps_dec[pos, tok]
    vec = vec @ W
    vec = vec + jnp.where(first_flag, ctx, 0.0)
    key = jax.random.fold_in(rng_key, 1000 + pos)
    vec = apply_dropout(vec, key, rate)
    logits = vec @ out_proj
    first_flag = False
    return (vec, ctx, mps_dec, out_proj, rng_key, rate, first_flag), logits

@jax.jit
def decode_teacher_forcing(params, thought_vector, decoder_input, rng_key, dropout_rate):
    L = params['decoder']['mps'].shape[0]
    bond_dim_mps = params['decoder']['mps'].shape[-1]
    ctx = thought_vector @ params['projection_matrix']
    vec0 = jnp.zeros((bond_dim_mps,), dtype=jnp.float32).at[0].set(1.0)
    positions = jnp.arange(L-1, dtype=jnp.int32)
    mps_dec = params['decoder']['mps'][:-1]
    (_, _, _, _, _, _, _), logits = lax.scan(
        _decode_step,
        (vec0, ctx, mps_dec, params['output_projection'], rng_key, dropout_rate, True),
        (decoder_input, positions)
    )
    return logits  # (L-1, V)

# ---------------------- Symbolic pattern tooling ------------------------
class SymbolicsHelper:
    """
    Lightweight symbolic signals + neighbor graph for a *subset* of sequences.
    - Builds only for the sequences you pass (no giant global arrays).
    - Masks default to ones (no hard forbids) to avoid aggressive constraints.
    """
    def __init__(self, processor, config: Dict):
        self.proc = processor
        self.config = dict(config or {})
        self.alphabet = self.config.get('alphabet', 'protein')
        self.k_neighbors = int(self.config.get('k_neighbors', 8))
        self.symbol_defs = list(self.config.get('symbols', []))
        self.symbol_ids = [d['id'] for d in self.symbol_defs]
        self.compiled = [re.compile(d['regex']) for d in self.symbol_defs]

    @property
    def num_symbols(self) -> int:
        return len(self.symbol_ids)

    def build_for_subset(self, seq_list: List[List[str]], max_len: int):
        """Return (masks_u8, symv_u8, neighbors_local) for the provided seqs."""
        V = int(self.proc.word_count)
        Lm1 = int(max_len - 1)
        S = self.num_symbols
        B = len(seq_list)

        masks = np.ones((B, Lm1, V), dtype=np.uint8)
        symv  = np.zeros((B, S), dtype=np.uint8) if S > 0 else np.zeros((B,1), np.uint8)

        strings = [''.join(s) for s in seq_list]
        # symbol presence
        for i, st in enumerate(strings):
            for s_idx, rgx in enumerate(self.compiled):
                if rgx.search(st) is not None:
                    symv[i, s_idx] = 1

        # neighbor graph from co-membership
        if S > 0 and self.k_neighbors > 0:
            idxs_by_symbol: Dict[int, List[int]] = {s: [] for s in range(S)}
            for i, st in enumerate(strings):
                for s_idx, rgx in enumerate(self.compiled):
                    if rgx.search(st) is not None:
                        idxs_by_symbol[s_idx].append(i)
            neighbor_sets = [set() for _ in range(B)]
            for s, members in idxs_by_symbol.items():
                if len(members) <= 1:
                    continue
                for pos, i in enumerate(members):
                    for off in range(1, min(self.k_neighbors, len(members)-1) + 1):
                        j = members[(pos + off) % len(members)]
                        if j != i: neighbor_sets[i].add(j)
            K = int(self.k_neighbors)
            neighbors_local = np.full((B, K), -1, dtype=np.int32)
            for i, sset in enumerate(neighbor_sets):
                if not sset: continue
                take = list(sset)[:K]
                neighbors_local[i, :len(take)] = np.array(take, dtype=np.int32)
        else:
            neighbors_local = np.full((B, 1), -1, dtype=np.int32)

        return masks, symv, neighbors_local

# ---------------------- Train-time utilities ------------------------

def ensure_symbolic_params(params: Dict, num_symbols: int, rng_key) -> Dict:
    """Ensure params['symbolic_tv_proj'] exists (shape: [S, thought_dim])."""
    if num_symbols <= 0:
        return params
    thought_dim = int(params['projection_matrix'].shape[0])
    need_init = ('symbolic_tv_proj' not in params) or (params['symbolic_tv_proj'].shape != (num_symbols, thought_dim))
    if need_init:
        k = jax.random.fold_in(rng_key, 777)
        params['symbolic_tv_proj'] = jax.random.normal(k, (num_symbols, thought_dim)) * 0.05
    return params

@jax.jit
def _xent_logits_with_int_labels(logits, labels_int, label_smoothing=0.0):
    V = logits.shape[-1]  # make num_classes concrete for JAX
    ls = jnp.asarray(label_smoothing, dtype=logits.dtype)
    onehot = jax.nn.one_hot(labels_int, V)
    smoothed = optax.smooth_labels(onehot, ls)
    return optax.softmax_cross_entropy(logits=logits, labels=smoothed)

@jax.jit
def loss_fn_symbolic(params: Dict,
                     batch_inputs: jnp.ndarray,
                     pad_idx: int,
                     rng_key,
                     dropout_rate: float,
                     label_smoothing: float,
                     allowed_masks_uint8: jnp.ndarray,   # (B, L-1, V) uint8
                     symbol_vecs_uint8: jnp.ndarray,     # (B, S)    uint8
                     pairs_padded: jnp.ndarray,          # (M, 2)    int32
                     pairs_mask: jnp.ndarray,            # (M,)      int32 {0,1}
                     lambda_graph: float = 0.0,
                     alpha: float = 1.0,
                     tau: float = 1.0) -> jnp.ndarray:
    """
    Total loss = CE + soft-constraint penalty + lambda_graph * Laplacian(tv).
      - tv := encode(seq) + alpha * (symv @ symbolic_tv_proj)
      - soft-constraint: encourage probability mass on allowed tokens
        with softening factor tau in [0,1]:
          allowed_soft = 1 - (1 - allowed) * tau
        tau=1 -> strict original mask; tau=0 -> all-ones (no penalty)
    """
    B = batch_inputs.shape[0]

    # Encode
    enc_keys = jax.random.split(rng_key, B)
    dec_keys = jax.random.split(jax.random.fold_in(rng_key, 12345), B)
    tvs = jax.vmap(encode, in_axes=(None,0,0,None))(
        params['encoder'], batch_inputs, enc_keys, dropout_rate
    )  # (B, thought_dim)

    # Add symbolic projection (if present)
    sv = symbol_vecs_uint8.astype(jnp.float32)
    if 'symbolic_tv_proj' in params:
        alpha_f = jnp.asarray(alpha, dtype=tvs.dtype)
        tvs = tvs + alpha_f * (sv @ params['symbolic_tv_proj'])

    # Decode
    dec_in  = batch_inputs[:, :-1]
    dec_tgt = batch_inputs[:, 1:]
    logits  = jax.vmap(decode_teacher_forcing, in_axes=(None,0,0,0,None))(
        params, tvs, dec_in, dec_keys, dropout_rate
    )  # (B, L-1, V)

    # CE
    xent = _xent_logits_with_int_labels(logits, dec_tgt, label_smoothing)  # (B,L-1)
    mask_tok = (dec_tgt != pad_idx)

    # Soft rule-penalty
    probs = jax.nn.softmax(logits, axis=-1)
    allowed = allowed_masks_uint8.astype(probs.dtype)
    tau_f = jnp.clip(jnp.asarray(tau, dtype=probs.dtype), 0.0, 1.0)
    allowed_soft = 1.0 - (1.0 - allowed) * tau_f
    allowed_mass = (probs * allowed_soft).sum(axis=-1)  # (B,L-1)
    rule_pen = 1.0 - allowed_mass

    token_losses = xent + rule_pen
    per_seq_sum = (token_losses * mask_tok).sum(axis=1)
    per_seq_cnt = mask_tok.sum(axis=1)
    token_loss = (per_seq_sum.sum() / jnp.maximum(per_seq_cnt.sum(), 1.0))

    # Graph Laplacian on tvs (pairs within batch)
    i = pairs_padded[:,0]; j = pairs_padded[:,1]
    valid = (pairs_mask > 0).astype(tvs.dtype)
    i = jnp.where(valid > 0, i, 0)
    j = jnp.where(valid > 0, j, 0)
    diffs = tvs[i] - tvs[j]
    gl = ((diffs**2).sum(axis=1) * valid).sum() / jnp.maximum(valid.sum(), 1.0)

    return token_loss + (jnp.asarray(lambda_graph, dtype=tvs.dtype) * gl)

def make_train_step_symbolic(optimizer):
    """Single-device train step."""
    @jax.jit
    def step(params: Dict, opt_state,
             batch_inputs: jnp.ndarray,
             pad_idx: int,
             rng_key,
             dropout_rate: float,
             label_smoothing: float,
             allowed_masks_uint8: jnp.ndarray,
             symbol_vecs_uint8: jnp.ndarray,
             pairs_padded: jnp.ndarray,
             pairs_mask: jnp.ndarray,
             lambda_graph: float = 0.0,
             alpha: float = 1.0,
             tau: float = 1.0):
        def _loss(p):
            return loss_fn_symbolic(
                p, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing,
                allowed_masks_uint8, symbol_vecs_uint8, pairs_padded, pairs_mask,
                lambda_graph, alpha, tau
            )
        l, grads = jax.value_and_grad(_loss)(params)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state, l
    return step

def make_train_step_symbolic_pmap(optimizer, axis_name: str = "dp"):
    """
    Multi-GPU train step using pmap (data-parallel).
    - params/opt_state are broadcast (in_axes=None)
    - batch and per-batch extras are sharded on leading axis (in_axes=0)
    - grads and loss are averaged with lax.pmean across devices
    - returns unsharded params/opt_state/loss (out_axes=None) for easy checkpointing
    """
    def step_impl(params: Dict, opt_state,
                  batch_inputs: jnp.ndarray,
                  pad_idx: int,
                  rng_key,                       # per-device key
                  dropout_rate: float,
                  label_smoothing: float,
                  allowed_masks_uint8: jnp.ndarray,
                  symbol_vecs_uint8: jnp.ndarray,
                  pairs_padded: jnp.ndarray,
                  pairs_mask: jnp.ndarray,
                  lambda_graph: float = 0.0,
                  alpha: float = 1.0,
                  tau: float = 1.0):
        def _loss(p):
            return loss_fn_symbolic(
                p, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing,
                allowed_masks_uint8, symbol_vecs_uint8, pairs_padded, pairs_mask,
                lambda_graph, alpha, tau
            )
        l, grads = jax.value_and_grad(_loss)(params)
        grads = lax.pmean(grads, axis_name=axis_name)
        l = lax.pmean(l, axis_name=axis_name)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state, l

    pstep = jax.pmap(
        step_impl,
        axis_name=axis_name,
        in_axes=(None, None, 0, None, 0, None, None, 0, 0, 0, 0, None, None, None),
        out_axes=(None, None, None),
    )
    return pstep

# --------------- batch-edge builder (graph pairs per batch) --------------
def get_pairs_for_batch(batch_indices: np.ndarray,
                        neighbors_local: np.ndarray,
                        edges_per_node: int = 4,
                        max_pairs: int = 2048) -> Tuple[jnp.ndarray, jnp.ndarray]:
    """
    Build a fixed-size (max_pairs,2) set of (i,j) edges within the batch for Laplacian reg.
    neighbors_local is (N_split, K) with local indices for the *split*.
    """
    loc_map = {int(j): b for b, j in enumerate(batch_indices)}
    pairs = []
    for b, j in enumerate(batch_indices):
        cnt = 0
        for nbj in neighbors_local[int(j)]:
            if nbj < 0: continue
            if int(nbj) in loc_map:
                pairs.append([b, loc_map[int(nbj)]])
                cnt += 1
                if cnt >= edges_per_node: break
        if len(pairs) >= max_pairs:
            break
    pairs_padded = np.zeros((max_pairs, 2), dtype=np.int32)
    pairs_mask   = np.zeros((max_pairs,), dtype=np.int32)
    if pairs:
        k = min(len(pairs), max_pairs)
        pairs_padded[:k] = np.array(pairs[:k], dtype=np.int32)
        pairs_mask[:k] = 1
    return jnp.asarray(pairs_padded), jnp.asarray(pairs_mask)


# Training prov1 me full resume

In [ ]:
%%writefile /kaggle/working/train_with_symbolics.py
# -*- coding: utf-8 -*-

import os, time, pickle, random, re, math
import numpy as np
import jax, jax.numpy as jnp
from jax import lax
import optax
from typing import List, Tuple, Dict, Optional

from symbolics_integration import (
    SymbolicsHelper, ensure_symbolic_params,
    make_train_step_symbolic, make_train_step_symbolic_pmap, get_pairs_for_batch,
    DEFAULT_PROTEIN_CONFIG,
)

# ===================== TUNABLE SETTINGS (YOUR PRESET) =====================
DATA_FILES: List[str]   = ["/kaggle/input/pfaseed/Pfam-A.seed"]

# Data / curriculum
MIN_LEN_LIMIT: int      = 10             # inclusive lower bound for corpus filtering
MAX_LEN_LIMIT: int      = 52            # EVEN, <= FIXED_MAX_LEN_CAP   66   pame->22
FIXED_MAX_LEN_CAP: int  = 256           # EVEN, parsing hard cap
AUTO_MAX_STAGES: int    = 8
MIN_BUCKET_FOR_STAGE: int = 300
FRAC_MODE: str          = "uniform"     # "uniform" or "by_tokens"
NUM_SAMPLES: int        = 20000         # target per curriculum (selected across stages)

# Model
BOND_DIMENSION_MPS: int = 32   #32
BOND_DIMENSION_MERA:int = 32   #32

# Train (dynamic tokens-per-batch scheduling)
TOKENS_PER_BATCH_TARGET = 8192
BATCH_SIZE = 80     #64
MIN_STAGE_BATCH: int    = 8             # floor per stage


#TOKENS_PER_BATCH_TARGET = 4096
#BATCH_SIZE = 32

MIN_STAGE_BATCH: int    = 8             # floor per stage

BASE_LR: float          = 2e-4
WEIGHT_DECAY: float     = 2e-4
DROPOUT_RATE_TRAIN:float= 0.08
LABEL_SMOOTHING: float  = 0.03
WARMUP_FRAC: float      = 0.10     #0.10
COSINE_FINAL_RATIO:float= 0.1
EPOCHS_PER_STAGE: int   = 800
MIN_EPOCHS: int         = 400   #800
PATIENCE: int           = 8
MIN_DELTA: float        = 1e-3

# ---------- Minimum validation PPL requirement ----------
# Set e.g. 9.2 to keep training (with upsampling) until best val-PPL <= 9.2
MIN_VAL_PPL_REQUIRED: Optional[float] = 30

# ---------- Multi-GPU ----------
USE_PMAP_MULTI_GPU: bool = True
PMAP_AXIS_NAME: str = "dp"  # data-parallel axis

# ---------- Checkpoint cadence ----------
# Save 'latest' every N epochs. Set 0 to disable periodic epoch saves.
SAVE_LATEST_EVERY: int = 15
# Always save 'latest' on early stop (even if not on SAVE_LATEST_EVERY boundary)
SAVE_LATEST_ON_EARLY_STOP: bool = True
# Always save 'latest' at the end of each stage
SAVE_LATEST_ON_STAGE_END: bool = True

# NEW: Best checkpoint cadence controls
# If False, never write 'best' to disk during epochs (only stage-end/early-stop flush if enabled below).
SAVE_BEST: bool = True
# If 0 => never write 'best' during epochs (buffer only; flush at stage end / early stop).
# If 1 => write immediately on each improvement (old behavior).
# If N>1 => buffer improvements and write only on epochs that are multiples of N.
SAVE_BEST_EVERY: int = 20
# Force-flush buffered best at early stop / stage end:
FLUSH_BEST_ON_EARLY_STOP: bool = True
FLUSH_BEST_ON_STAGE_END: bool  = True

# Symbolics
USE_SYMBOLICS = True
GRAPH_EDGES_PER_NODE = 4
GRAPH_MAX_PAIRS = 2048
LAMBDA_GRAPH_BASE = 0.003   # start small and ramp

# Adaptive upsampling (Balanced-HEM)
ADAPTIVE_UPSAMPLING: bool = True
UPSAMPLE_TRIGGER_PATIENCE: int = 2   #4
MAX_UPSAMPLE_EVENTS_PER_STAGE: int = 6 #10
COOLDOWN_EPOCHS: int = 8   #80
REL_IMPROVEMENT_WINDOW: int = 8    #20
MIN_REL_IMPROVEMENT: float = 0.02    #0.02
PPL_ABS_CEILING: Optional[float] = None

UPSAMPLE_FACTOR_BASE: float = 1.30
UPSAMPLE_FACTOR_STEP: float = 0.10
MAX_UPSAMPLE_FACTOR: float = 1.80
REPEAT_CAP_MULTIPLIER: float = 2.0
UPSAMPLED_DROPOUT: float = 0.18
UPSAMPLED_SMOOTH: float = 0.08

# Hard-Example Mining
ENABLE_HEM: bool = True
HEM_ON_UPSAMPLE: bool = True
HEM_CANDIDATE_FRAC: float = 0.20
HEM_CANDIDATE_MAX: int = 8000
HEM_EVAL_TOKENS_PER_BATCH: int = 8192

# RNG / Paths
RNG_KEY = jax.random.PRNGKey(0)
PY_SEED = 0
CHECKPOINT_DIR = '/kaggle/working/'
MODEL_FILE     = '/kaggle/working/production_model_bio.pkl'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ===================== Data Processor =====================
class DataProcessor:
    def __init__(self, sequence_type='protein'):
        if sequence_type == 'protein':
            self.valid_chars = set('ACDEFGHIKLMNPQRSTVWY')
        elif sequence_type == 'dna':
            self.valid_chars = set('ACGT')
        elif sequence_type == 'rna':
            self.valid_chars = set('ACGU')
        else:
            raise ValueError("Unsupported sequence_type.")
        self.special_tokens = ['<PAD>','<SOS>','<EOS>','<UNK>']
        self.vocab, self.inv_vocab = {}, {}
        self.word_count = 0

    def build_vocab(self):
        for i, tok in enumerate(self.special_tokens):
            self.vocab[tok] = i; self.inv_vocab[i] = tok
        idx = len(self.special_tokens)
        for ch in sorted(list(self.valid_chars)):
            self.vocab[ch] = idx; self.inv_vocab[idx] = ch; idx += 1
        self.word_count = len(self.vocab)

    def _is_seq_line(self, line: str) -> bool:
        if (not line) or line.startswith('#') or line.startswith('//'):
            return False
        parts = line.split()
        if len(parts) < 2: return False
        seq = parts[-1]
        return bool(re.fullmatch(r'[A-Za-z\-\.]+', seq))

    def load_stockholm_ungapped(self, file_paths: List[str], min_len=1, max_len_cap=256):
        all_seqs: List[List[str]] = []
        for path in file_paths:
            if not os.path.exists(path):
                print(f"⚠️  File not found: {path}. Skipping.")
                continue
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    for raw in f:
                        line = raw.strip()
                        if not self._is_seq_line(line):
                            continue
                        parts = line.split()
                        seq = parts[-1].upper()
                        seq = ''.join([c for c in seq if c not in ('.','-') and c in self.valid_chars])
                        if len(seq) >= min_len:
                            seq = seq[:max_len_cap-2]  # -2 for SOS/EOS
                            all_seqs.append(list(seq))
            except Exception as e:
                print(f"⚠️  Failed to read {path}: {e}")
                continue
        # deduplicate
        uniq, seen = [], set()
        for lst in all_seqs:
            s = ''.join(lst)
            if s not in seen:
                seen.add(s); uniq.append(lst)
        return uniq

    def sequence_to_indices(self, seq_chars: List[str]):
        unk = self.vocab['<UNK>']
        return np.array([self.vocab.get(c, unk) for c in seq_chars], dtype=np.int32)

# ===================== Model / Init =====================

def init_single_mera_compact(key, vocab_size, L, chi_mps, chi_mera):
    k_mps, k_iso = jax.random.split(key)
    mps = jax.random.normal(k_mps, (L, vocab_size, chi_mps, chi_mps)) * 0.1
    isos = jax.random.normal(k_iso, (L//2, chi_mps, chi_mps, chi_mera)) * 0.1
    return {"mps": mps, "isos": isos}

def init_encoder_decoder_model(key, physical_dim, bond_dim_mps, bond_dim_mera, max_len):
    assert max_len % 2 == 0, "MAX_LEN must be even for MERA pairwise reduction"
    k1,k2,k3,k4 = jax.random.split(key,4)
    enc = init_single_mera_compact(k1, physical_dim, max_len, bond_dim_mps, bond_dim_mera)
    dec = init_single_mera_compact(k2, physical_dim, max_len, bond_dim_mps, bond_dim_mera)
    thought_dim = (max_len//2) * bond_dim_mera
    projection_matrix = jax.random.normal(k3, (thought_dim, bond_dim_mps)) * 0.1
    output_projection = jax.random.normal(k4, (bond_dim_mps, physical_dim)) * 0.1
    return {
        'encoder': enc,
        'decoder': dec,
        'projection_matrix': projection_matrix,
        'output_projection': output_projection
    }

# ===================== Dropout helper =====================

def apply_dropout(x, key, rate):
    rate = jnp.asarray(rate, dtype=x.dtype)
    keep_prob = jnp.clip(1.0 - rate, 0.0, 1.0)
    def do_keep(_):
        mask = jax.random.bernoulli(key, p=keep_prob, shape=x.shape)
        return jnp.where(mask, x / jnp.maximum(keep_prob, 1e-6), 0.0)
    return lax.cond(jnp.equal(rate, 0.0), lambda _: x, do_keep, operand=None)

# ===================== Encode / Decode =====================

def _encode_step(carry, token_and_pos):
    state, mps, rng_key, rate = carry
    tok, pos = token_and_pos
    W = mps[pos, tok]
    state = state @ W
    key = jax.random.fold_in(rng_key, pos)
    state = apply_dropout(state, key, rate)
    return (state, mps, rng_key, rate), state

@jax.jit
def encode(enc_params, seq_indices, rng_key, dropout_rate):
    L = enc_params['mps'].shape[0]
    assert L % 2 == 0, "Encoder length must be even for MERA pairwise reduction"
    bond_dim_mps = enc_params['mps'].shape[-1]
    state0 = jnp.zeros((bond_dim_mps,), dtype=jnp.float32).at[0].set(1.0)
    positions = jnp.arange(L, dtype=jnp.int32)
    (_, _, _, _), states = lax.scan(
        _encode_step,
        (state0, enc_params['mps'], rng_key, dropout_rate),
        (seq_indices, positions)
    )
    # MERA pairwise reduction
    def pair_reduce(_, i):
        l = states[2*i]
        r = states[2*i + 1]
        iso = enc_params['isos'][i]
        hi = jnp.einsum("i,j,ijk->k", l, r, iso)
        return None, hi
    _, hi_list = lax.scan(pair_reduce, None, jnp.arange(L//2))
    tv = hi_list.reshape(-1)
    return tv / (jnp.linalg.norm(tv) + 1e-9)

def _decode_step(carry, tok_and_pos):
    vec, ctx, mps_dec, out_proj, rng_key, rate, first_flag = carry
    tok, pos = tok_and_pos
    W = mps_dec[pos, tok]
    vec = vec @ W
    vec = vec + jnp.where(first_flag, ctx, 0.0)
    key = jax.random.fold_in(rng_key, 1000 + pos)
    vec = apply_dropout(vec, key, rate)
    logits = vec @ out_proj
    first_flag = False
    return (vec, ctx, mps_dec, out_proj, rng_key, rate, first_flag), logits

@jax.jit
def decode_teacher_forcing(params, thought_vector, decoder_input, rng_key, dropout_rate):
    L = params['decoder']['mps'].shape[0]
    bond_dim_mps = params['decoder']['mps'].shape[-1]
    ctx = thought_vector @ params['projection_matrix']
    vec0 = jnp.zeros((bond_dim_mps,), dtype=jnp.float32).at[0].set(1.0)
    positions = jnp.arange(L-1, dtype=jnp.int32)
    mps_dec = params['decoder']['mps'][:-1]
    (_, _, _, _, _, _, _), logits = lax.scan(
        _decode_step,
        (vec0, ctx, mps_dec, params['output_projection'], rng_key, dropout_rate, True),
        (decoder_input, positions)
    )
    return logits  # (L-1, V)

# ===================== Loss / Train =====================

@jax.jit
def _xent_logits_with_int_labels(logits, labels_int, label_smoothing=0.0):
    V = logits.shape[-1]
    ls = jnp.asarray(label_smoothing, dtype=logits.dtype)
    onehot = jax.nn.one_hot(labels_int, V)
    smoothed = optax.smooth_labels(onehot, ls)
    return optax.softmax_cross_entropy(logits=logits, labels=smoothed)

@jax.jit
def per_seq_losses(params, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing):
    B = batch_inputs.shape[0]
    enc_keys = jax.random.split(rng_key, B)
    dec_keys = jax.random.split(jax.random.fold_in(rng_key, 12345), B)
    tvs = jax.vmap(encode, in_axes=(None,0,0,None))(
        params['encoder'], batch_inputs, enc_keys, dropout_rate
    )
    dec_in  = batch_inputs[:, :-1]
    dec_tgt = batch_inputs[:, 1:]
    logits  = jax.vmap(decode_teacher_forcing, in_axes=(None,0,0,0,None))(
        params, tvs, dec_in, dec_keys, dropout_rate
    )  # (B,L-1,V)
    token_losses = _xent_logits_with_int_labels(logits, dec_tgt, label_smoothing)
    mask = (dec_tgt != pad_idx)
    per_seq_sum = (token_losses * mask).sum(axis=1)
    per_seq_cnt = mask.sum(axis=1)
    return per_seq_sum / jnp.maximum(per_seq_cnt, 1)

@jax.jit
def loss_fn(params, batch_inputs, batch_targets, pad_idx, rng_key, dropout_rate, label_smoothing):
    losses = per_seq_losses(params, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing)
    counts = (batch_targets[:, 1:] != pad_idx).sum(axis=1).astype(jnp.float32)
    denom = jnp.maximum(counts.sum(), 1.0)
    return (losses * counts).sum() / denom

# ===================== Helpers =====================

def set_pad_identity(params, pad_idx):
    mps_enc = params['encoder']['mps']
    mps_dec = params['decoder']['mps']
    L, V, chi, _ = mps_enc.shape
    I = jnp.eye(chi, dtype=mps_enc.dtype)
    mps_enc = mps_enc.at[:, pad_idx].set(I)
    mps_dec = mps_dec.at[:, pad_idx].set(I)
    params['encoder']['mps'] = mps_enc
    params['decoder']['mps'] = mps_dec

def pad_and_pack(processor, seq_chars, max_len):
    sos=processor.vocab['<SOS>']; eos=processor.vocab['<EOS>']; pad=processor.vocab['<PAD>']
    seq = ['<SOS>'] + seq_chars[:max_len-2] + ['<EOS>']
    idx = processor.sequence_to_indices(seq)
    if len(idx) < max_len:
        idx = np.concatenate([idx, np.full((max_len-len(idx),), pad, np.int32)])
    return idx

def build_arrays(processor, seq_lists, max_len):
    if len(seq_lists) == 0:
        return np.empty((0, max_len), dtype=np.int32)
    return np.asarray([pad_and_pack(processor, s, max_len) for s in seq_lists], dtype=np.int32)

@jax.jit
def eval_loss_jit(params, data_arr, pad_idx, rng_key, label_smoothing):
    l = loss_fn(params, data_arr, data_arr, pad_idx, rng_key, 0.0, label_smoothing)
    return l

def compute_perplexity(params, data_arr, pad_idx, batch=64, label_smoothing=0.0):
    n = len(data_arr)
    if n == 0:
        return float('inf')
    batch = max(1, min(batch, n))
    rng = jax.random.PRNGKey(999)
    tot_ll = 0.0; tot_tokens = 0
    for i in range(0, n, batch):
        chunk_np = np.asarray(data_arr[i:i+batch])
        chunk = jnp.asarray(chunk_np)
        l = float(eval_loss_jit(params, chunk, pad_idx, rng, label_smoothing))
        nonpad = int((chunk_np[:, 1:] != pad_idx).sum())
        tot_ll += l * nonpad
        tot_tokens += nonpad
    if tot_tokens == 0:
        return float('inf')
    return float(np.exp(tot_ll / tot_tokens))

def atomic_pickle_save(obj, path):
    tmp = path + ".tmp"
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    with open(path, 'rb') as f:
        _ = pickle.load(f)

# ---------- Dataset persistence helpers ----------
def seq_key(chars_list: List[str]) -> str:
    return ''.join(chars_list)

def rebuild_stage_extra(all_sequences: List[List[str]], bounds: Tuple[int,int], used_keys_set: set):
    low, high = bounds
    candidates = [s for s in all_sequences if low <= len(s) <= high]
    extra = [s for s in candidates if seq_key(s) not in used_keys_set]
    return extra

# ---------- Checkpoint helpers ----------

def stage_ckpt_path(ckpt_dir: str, max_len: int) -> str:
    return os.path.join(ckpt_dir, f"latest_len{max_len}.pkl")

def best_stage_ckpt_path(ckpt_dir: str, max_len: int) -> str:
    return os.path.join(ckpt_dir, f"best_len{max_len}.pkl")

def try_load_checkpoint(path: str):
    try:
        with open(path, 'rb') as f:
            ch = pickle.load(f)
        return ch
    except Exception:
        return None

def load_checkpoint_for_len(ckpt_dir: str, max_len: int):
    stage_path = stage_ckpt_path(ckpt_dir, max_len)
    ch = try_load_checkpoint(stage_path)
    if ch is not None:
        return ch, stage_path
    generic = os.path.join(ckpt_dir, 'latest.pkl')
    ch = try_load_checkpoint(generic)
    if ch is not None:
        return ch, generic
    return None, None

def save_checkpoint_dual(obj, ckpt_dir: str, max_len: int, is_best: bool=False):
    generic = os.path.join(ckpt_dir, 'latest.pkl')
    stage   = stage_ckpt_path(ckpt_dir, max_len)
    atomic_pickle_save(obj, generic)
    atomic_pickle_save(obj, stage)
    if is_best:
        best_p = best_stage_ckpt_path(ckpt_dir, max_len)
        atomic_pickle_save(obj, best_p)
    size_mb = os.path.getsize(generic)/1024/1024
    tag = " + best" if is_best else ""
    print(f"💾 Checkpoint saved & verified: {generic} ({size_mb:.2f} MB)  | mirror: {stage}{tag}")

# ---------- Grow / Shrink params when max_len changes ----------

def grow_params(params, old_len, new_len, bond_dim_mps, bond_dim_mera, vocab_size, rng):
    assert new_len > old_len and new_len % 2 == 0
    def grow_block(block, rng_block):
        mps_old = block['mps']; isos_old = block['isos']
        add_L = new_len - old_len
        k1, k2 = jax.random.split(rng_block)
        mps_new = jnp.zeros((new_len, vocab_size, bond_dim_mps, bond_dim_mps), dtype=mps_old.dtype)
        mps_new = mps_new.at[:old_len].set(mps_old * 0.999)
        if add_L > 0:
            init_tail = jax.random.normal(k1, (add_L, vocab_size, bond_dim_mps, bond_dim_mps)) * 0.1
            mps_new = mps_new.at[old_len:].set(init_tail)
        old_d = old_len // 2; new_d = new_len // 2
        isos_new = jnp.zeros((new_d, bond_dim_mps, bond_dim_mps, bond_dim_mera), dtype=isos_old.dtype)
        isos_new = isos_new.at[:old_d].set(isos_old * 0.999)
        if new_d - old_d > 0:
            init_iso_tail = jax.random.normal(k2, (new_d - old_d, bond_dim_mps, bond_dim_mps, bond_dim_mera)) * 0.1
            isos_new = isos_new.at[old_d:].set(init_iso_tail)
        return {'mps': mps_new, 'isos': isos_new}
    rng_enc, rng_dec, kproj = jax.random.split(rng, 3)
    new_encoder = grow_block(params['encoder'], rng_enc)
    new_decoder = grow_block(params['decoder'], rng_dec)
    old_td = (old_len//2) * bond_dim_mera
    new_td = (new_len//2) * bond_dim_mera
    proj_old = params['projection_matrix']
    proj_new = jnp.zeros((new_td, bond_dim_mps), dtype=proj_old.dtype)
    k = min(old_td, new_td)
    proj_new = proj_new.at[:k, :].set(proj_old[:k, :])
    if new_td > k:
        init_tail = jax.random.normal(kproj, (new_td - k, bond_dim_mps)) * 0.1
        proj_new = proj_new.at[k:, :].set(init_tail)
    return {
        'encoder': new_encoder,
        'decoder': new_decoder,
        'projection_matrix': proj_new,
        'output_projection': params['output_projection'],
    }

def shrink_params(params, old_len, new_len, bond_dim_mps, bond_dim_mera):
    assert new_len < old_len and new_len % 2 == 0
    def shrink_block(block):
        mps_old = block['mps']; isos_old = block['isos']
        return {'mps': mps_old[:new_len], 'isos': isos_old[:(new_len//2)]}
    new_encoder = shrink_block(params['encoder'])
    new_decoder = shrink_block(params['decoder'])
    old_td = (old_len//2) * bond_dim_mera
    new_td = (new_len//2) * bond_dim_mera
    proj_old = params['projection_matrix']
    proj_new = proj_old[:new_td]
    return {
        'encoder': new_encoder,
        'decoder': new_decoder,
        'projection_matrix': proj_new,
        'output_projection': params['output_projection'],
    }

# ===================== Dynamic staging from corpus =====================

def length_histogram(sequences: List[List[str]], max_len_limit: int) -> List[int]:
    counts = [0]*(max_len_limit+1)
    for s in sequences:
        L = len(s)
        if 1 <= L <= max_len_limit:
            counts[L] += 1
    return counts

def greedy_stage_endpoints(counts: List[int], max_len_limit: int, target_per_stage: int) -> List[int]:
    endpoints = []; acc = 0
    for L in range(1, max_len_limit+1):
        acc += counts[L]
        if acc >= target_per_stage and (L % 2 == 0):
            endpoints.append(L); acc = 0
    last_even = max_len_limit if (max_len_limit % 2 == 0) else (max_len_limit-1)
    if not endpoints or endpoints[-1] != last_even:
        if last_even > 0 and (not endpoints or endpoints[-1] < last_even):
            endpoints.append(last_even)
    cleaned = []; prev = 0
    for L in endpoints:
        if L > prev and L % 2 == 0:
            cleaned.append(L); prev = L
    return cleaned

def build_dynamic_stages_from_corpus(sequences: List[List[str]], max_len_limit: int,
                                     auto_max_stages: int, min_bucket_for_stage: int) -> List[int]:
    assert max_len_limit % 2 == 0
    counts = length_histogram(sequences, max_len_limit)
    total = sum(counts[1:])
    if total == 0:
        return [max_len_limit]
    tgt = max(min_bucket_for_stage, math.ceil(total / max(1, auto_max_stages)))
    for _ in range(10):
        endpoints = greedy_stage_endpoints(counts, max_len_limit, tgt)
        if len(endpoints) <= auto_max_stages:
            return endpoints
        tgt = int(tgt * 1.25)
    return greedy_stage_endpoints(counts, max_len_limit, tgt)

def make_stage_bounds(length_stages: List[int], start_min_len: int = 1) -> List[Tuple[int,int]]:
    bounds = []; prev = max(1, int(start_min_len) - 1)
    for L in length_stages:
        bounds.append((prev+1, L)); prev = L
    return bounds

def stage_stats(all_sequences: List[List[str]], bounds_list: List[Tuple[int,int]]) -> Tuple[List[int], List[int], List[float]]:
    counts = [0]*len(bounds_list); tok_sums = [0]*len(bounds_list)
    for s in all_sequences:
        L = len(s)
        for i, (low, high) in enumerate(bounds_list):
            if low <= L <= high:
                counts[i] += 1; tok_sums[i] += L; break
    means = [ (tok_sums[i]/counts[i]) if counts[i]>0 else 0.0 for i in range(len(bounds_list)) ]
    return counts, tok_sums, means

def compute_stage_fractions(bounds_list: List[Tuple[int,int]], counts: List[int], tok_sums: List[int], mode: str = "by_tokens") -> List[float]:
    if mode not in ("uniform", "by_tokens"):
        mode = "by_tokens"
    raw = [1.0 if c>0 else 0.0 for c in counts] if mode=="uniform" else [float(tok) for tok in tok_sums]
    s = sum(raw)
    if s <= 0:
        return [1.0/len(bounds_list)]*len(bounds_list)
    return [x/s for x in raw]

def compute_targets(num_samples: int, fracs: List[float], avail: List[int]) -> List[int]:
    assert len(fracs) == len(avail)
    total_avail = sum(avail)
    budget = min(int(num_samples), total_avail)
    if budget <= 0: return [0]*len(fracs)
    s = float(sum(fracs))
    fracs = [f/s for f in fracs] if s>0 else [1.0/len(fracs)]*len(fracs)
    raw = [budget * f for f in fracs]
    base = [min(int(x), a) for x,a in zip(raw, avail)]
    rema = [x - int(x) for x in raw]
    used = sum(base); left = budget - used
    order = sorted(range(len(fracs)), key=lambda i: rema[i], reverse=True)
    for i in order:
        if left<=0: break
        if base[i] < avail[i]:
            base[i]+=1; left-=1
    return base

def select_stage_sequences_no_upsample(all_sequences: List[List[str]], bounds: Tuple[int,int],
                                       target_count: int, seed: int) -> Tuple[List[List[str]], List[List[str]]]:
    low, high = bounds
    candidates = [s for s in all_sequences if low <= len(s) <= high]
    r = random.Random(seed); r.shuffle(candidates)
    k = min(target_count, len(candidates))
    selected = candidates[:k]; extra_pool = candidates[k:]
    return selected, extra_pool

# ===================== Hard-Example Mining helpers =====================

def compute_stage_batch_size(max_len: int) -> int:
    return max(MIN_STAGE_BATCH, min(BATCH_SIZE, max(1, TOKENS_PER_BATCH_TARGET // max(1, max_len))))

def compute_eval_batch_size(max_len: int, tpb: int) -> int:
    return max(1, min(BATCH_SIZE, max(1, tpb // max(1, max_len))))

def hard_mine_from_extra_pool(params, processor, extra_pool: List[List[str]],
                              max_len: int, pad_idx: int, curr_smoothing: float,
                              need: int, rng_seed: int) -> Tuple[List[List[str]], List[List[str]], Dict[str,int]]:
    if need <= 0 or len(extra_pool) == 0:
        return [], extra_pool, {'cands':0,'eval_batches':0,'picked':0}

    r = random.Random(rng_seed)
    n_cands = min(len(extra_pool), max(1, int(len(extra_pool) * HEM_CANDIDATE_FRAC)))
    n_cands = min(n_cands, HEM_CANDIDATE_MAX)
    cand_idx = r.sample(range(len(extra_pool)), n_cands)
    candidates = [extra_pool[i] for i in cand_idx]

    cand_arr = build_arrays(processor, candidates, max_len)
    eval_batch = compute_eval_batch_size(max_len, HEM_EVAL_TOKENS_PER_BATCH)
    losses = []
    rng = jax.random.PRNGKey(1337)
    for i in range(0, len(cand_arr), eval_batch):
        chunk = jnp.asarray(cand_arr[i:i+eval_batch])
        lvec = per_seq_losses(params, chunk, pad_idx, rng, 0.0, 0.0)
        losses.append(np.array(lvec))
    losses = np.concatenate(losses, axis=0) if losses else np.array([])

    k = min(need, len(candidates))
    top_idx_local = np.argsort(-losses)[:k]  # descending
    picked = [candidates[i] for i in top_idx_local]

    pick_global = set(cand_idx[i] for i in top_idx_local)
    new_extra = [s for i,s in enumerate(extra_pool) if i not in pick_global]

    stats = {'cands': n_cands, 'eval_batches': int(math.ceil(n_cands / eval_batch)), 'picked': len(picked)}
    return picked, new_extra, stats

def upsample_train_set_general(
    base_train: List[List[str]],
    extra_pool: List[List[str]],
    factor: float,
    repeat_cap_multiplier: float,
    seed: int,
    strategy: str,
    params=None, processor=None, max_len=None, pad_idx=None, curr_smoothing: float=0.0
) -> Tuple[List[List[str]], List[List[str]], Dict[str,int]]:
    r = random.Random(seed)
    n0 = len(base_train)
    if n0 == 0:
        return base_train, extra_pool, {'added_unique':0, 'added_repeats':0, 'target':0, 'final':0, 'strategy':strategy}

    max_allowed = int(max(1, n0 * repeat_cap_multiplier))
    desired = int(min(max_allowed, max(n0+1, n0 * factor)))

    new_train = list(base_train)
    new_extra = list(extra_pool)

    need = desired - len(new_train)
    added_unique = 0
    if need > 0 and len(new_extra) > 0:
        if strategy == 'hard' and ENABLE_HEM and HEM_ON_UPSAMPLE and params is not None:
            mined, new_extra, _ = hard_mine_from_extra_pool(
                params, processor, new_extra, max_len, pad_idx, curr_smoothing, need, seed+4242
            )
            new_train.extend(mined); added_unique += len(mined); need = desired - len(new_train)
        if need > 0 and len(new_extra) > 0:
            r.shuffle(new_extra)
            take = min(need, len(new_extra))
            new_train.extend(new_extra[:take]); new_extra = new_extra[take:]; added_unique += take

    need = desired - len(new_train)
    repeats_used = 0
    if need > 0:
        pool = list(new_train)
        for _ in range(need):
            new_train.append(r.choice(pool)); repeats_used += 1

    stats = {'added_unique': added_unique, 'added_repeats': repeats_used,
             'target': desired, 'final': len(new_train), 'strategy': strategy}
    return new_train, new_extra, stats

# ===================== Adaptive upsampling trigger =====================

def _rel_improve(old: float, new: float) -> float:
    if not (np.isfinite(old) and np.isfinite(new)) or old <= 0:
        return 0.0
    return max(0.0, (old - new) / old)

def should_adaptive_upsample(
    epoch_idx: int,
    val_hist: List[float],
    best_val: float,
    last_upsample_epoch: int,
    min_trigger_epoch: int,
    plateau_patience: int,
    window: int,
    min_rel_improve: float,
    abs_ceiling: Optional[float],
    cooldown_epochs: int,
    min_required: Optional[float] = None,   # continue nudging until <= min_required
) -> Tuple[bool, str]:
    """
    After an upsample, compute plateau/low_slope relative only to the history *since* that event.
    If min_required is set and we're still above it, demand a plateau (or abs_ceiling) to trigger,
    to avoid spamming back-to-back upsampling on mere low_slope signals.
    """
    e = epoch_idx + 1
    if e < min_trigger_epoch:
        return (False, "warmup")

    # Cooldown from the last upsample
    if last_upsample_epoch >= 0 and (e - last_upsample_epoch) < cooldown_epochs:
        return (False, "cooldown")

    anchor = max(0, last_upsample_epoch)
    hist_since = val_hist[anchor:] if anchor < len(val_hist) else []
    if len(hist_since) < 2:
        return (False, "insufficient_data")

    plateau = False
    if len(hist_since) >= plateau_patience + 1:
        tail = hist_since[-(plateau_patience+1):]
        improved = any(tail[i] + MIN_DELTA < tail[i-1] for i in range(1, len(tail)))
        plateau = not improved

    low_slope = False
    if len(hist_since) >= window + 1:
        before = hist_since[-(window+1)]
        now = hist_since[-1]
        rel = _rel_improve(before, now)
        low_slope = (rel < min_rel_improve)

    too_high = (abs_ceiling is not None and hist_since[-1] > abs_ceiling)
    above_min = (min_required is not None and hist_since[-1] > min_required)

    if above_min:
        trigger = plateau or too_high
        reason  = "plateau" if plateau else ("abs_ceiling" if too_high else "above_min_no_plateau")
    else:
        trigger = plateau or low_slope or too_high
        if plateau:
            reason = "plateau"
        elif low_slope:
            reason = "low_slope"
        elif too_high:
            reason = "abs_ceiling"
        else:
            reason = "none"

    return (trigger, reason)


# ===================== Schedules =====================

def make_optimizer(total_steps: int):
    warmup_steps = max(1, int(total_steps * WARMUP_FRAC))
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=BASE_LR,
        warmup_steps=warmup_steps,
        decay_steps=max(1, total_steps - warmup_steps),
        end_value=BASE_LR * COSINE_FINAL_RATIO,
    )
    opt = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=schedule, weight_decay=WEIGHT_DECAY)
    )
    return opt

# ===================== Train/Val/Test split =====================

def split_for_len(seqs, seed=42, ratios=(0.8,0.1,0.1)):
    r = random.Random(seed); seqs = list(seqs); r.shuffle(seqs)
    n=len(seqs); n_train=int(n*ratios[0]); n_val=int(n*ratios[1])
    return seqs[:n_train], seqs[n_train:n_train+n_val], seqs[n_train+n_val:]

# ===================== Multi-GPU helpers =====================

def shard_first_axis(x: np.ndarray, ndev: int):
    """Reshape [B, *] -> [ndev, per_dev, *]. Assumes B % ndev == 0."""
    B = x.shape[0]
    per = B // ndev
    new_shape = (ndev, per) + tuple(x.shape[1:])
    return x.reshape(new_shape)

def shard_pairs(per_device_pairs, per_device_masks):
    """Stack list of per-device (M,2)/(M,) into arrays [D,M,2] and [D,M]."""
    pairs = np.stack(per_device_pairs, axis=0)
    masks = np.stack(per_device_masks, axis=0)
    return pairs, masks

# ===================== Main =====================

def main():
    # Reproducibility
    random.seed(PY_SEED); np.random.seed(PY_SEED)

    # Devices
    N_DEV = jax.local_device_count()
    IS_MULTI = USE_PMAP_MULTI_GPU and (N_DEV > 1)
    print(f"🖥️  JAX devices: {N_DEV} | multi-GPU={'ON' if IS_MULTI else 'OFF'} (pmap)")

    # Checks
    assert FIXED_MAX_LEN_CAP % 2 == 0, "FIXED_MAX_LEN_CAP must be even"
    assert MAX_LEN_LIMIT % 2 == 0, "MAX_LEN_LIMIT must be even"
    assert 1 <= MIN_LEN_LIMIT <= MAX_LEN_LIMIT, "MIN_LEN_LIMIT must be within [1, MAX_LEN_LIMIT]"

    print("📦 Setup:")
    print(f"  MIN_LEN_LIMIT       = {MIN_LEN_LIMIT}")
    print(f"  MAX_LEN_LIMIT       = {MAX_LEN_LIMIT}")
    print(f"  NUM_SAMPLES         = {NUM_SAMPLES}")
    print(f"  AUTO_MAX_STAGES     = {AUTO_MAX_STAGES}")
    print(f"  MIN_BUCKET_FOR_STAGE= {MIN_BUCKET_FOR_STAGE}")
    print(f"  FRAC_MODE           = {FRAC_MODE}")
    print(f"  BATCH_SIZE(max)     = {BATCH_SIZE}")
    print(f"  TPB target          = {TOKENS_PER_BATCH_TARGET}")
    print(f"  Adaptive Upsampling = {ADAPTIVE_UPSAMPLING}, HEM={ENABLE_HEM}")
    print(f"  HEM candidates frac/max = {HEM_CANDIDATE_FRAC}/{HEM_CANDIDATE_MAX}")
    print(f"  MIN_VAL_PPL_REQUIRED= {MIN_VAL_PPL_REQUIRED}")
    print(f"  SAVE_LATEST_EVERY   = {SAVE_LATEST_EVERY} (0=off)")
    print(f"  SAVE_BEST_EVERY     = {SAVE_BEST_EVERY} (0=buffer-only, 1=immediate, N=periodic)")

    processor = DataProcessor(sequence_type='protein')
    sequences = processor.load_stockholm_ungapped(
        DATA_FILES, min_len=MIN_LEN_LIMIT, max_len_cap=FIXED_MAX_LEN_CAP
    )
    print(f"Full corpus size (deduped, len>={MIN_LEN_LIMIT}): {len(sequences)}")
    random.shuffle(sequences)

    processor.build_vocab()
    VOCAB_SIZE = processor.word_count
    print(f"Vocab size = {VOCAB_SIZE}")

    # ----- Dynamic stages -----
    LENGTH_STAGES = build_dynamic_stages_from_corpus(
        sequences,
        max_len_limit=MAX_LEN_LIMIT,
        auto_max_stages=AUTO_MAX_STAGES,
        min_bucket_for_stage=MIN_BUCKET_FOR_STAGE
    )
    print("Dynamic LENGTH_STAGES =", LENGTH_STAGES)
    
    # ➜ Force single stage όταν ζητάς 1 (ή 0) stages
    if AUTO_MAX_STAGES <= 1:
        last_even = MAX_LEN_LIMIT if (MAX_LEN_LIMIT % 2 == 0) else (MAX_LEN_LIMIT - 1)
        LENGTH_STAGES = [last_even]
    
    # Προαιρετικό ενδιάμεσο eval stage — OFF
    TARGET_EVAL_LEN = None
    if (TARGET_EVAL_LEN is not None
        and TARGET_EVAL_LEN % 2 == 0
        and MIN_LEN_LIMIT <= TARGET_EVAL_LEN <= MAX_LEN_LIMIT
        and TARGET_EVAL_LEN not in LENGTH_STAGES):
        LENGTH_STAGES = sorted(set(LENGTH_STAGES + [TARGET_EVAL_LEN]))
    
    print("Adjusted LENGTH_STAGES =", LENGTH_STAGES)


    bounds_list = make_stage_bounds(LENGTH_STAGES, start_min_len=MIN_LEN_LIMIT)
    counts, tok_sums, mean_lens = stage_stats(sequences, bounds_list)

    # Prune empty stages
    if any(c == 0 for c in counts):
        pruned = [LENGTH_STAGES[i] for i,c in enumerate(counts) if c == 0]
        print("ℹ️ Pruning empty stages:", pruned)
        LENGTH_STAGES = [LENGTH_STAGES[i] for i,c in enumerate(counts) if c > 0]
        bounds_list = make_stage_bounds(LENGTH_STAGES, start_min_len=MIN_LEN_LIMIT)
        counts, tok_sums, mean_lens = stage_stats(sequences, bounds_list)

    fracs = compute_stage_fractions(bounds_list, counts, tok_sums, mode=FRAC_MODE)
    per_stage_targets = compute_targets(NUM_SAMPLES, fracs, counts)

    # Build per-stage selected + extra_pool
    stages_data = []
    for i, (bounds, target) in enumerate(zip(bounds_list, per_stage_targets)):
        stage_seed = PY_SEED + 1000 + i
        selected, extra_pool = select_stage_sequences_no_upsample(sequences, bounds, target, stage_seed)
        stages_data.append({'bounds': bounds, 'selected': selected, 'extra_pool': extra_pool})
        print(f"Stage {i+1} [{bounds[0]},{bounds[1]}]: avail={counts[i]}, target={target}, selected={len(selected)}, extra_pool={len(extra_pool)}")

    total_selected = sum(len(s['selected']) for s in stages_data)
    if total_selected < NUM_SAMPLES:
        print(f"ℹ️ Selected {total_selected} < requested {NUM_SAMPLES} (no upsampling at allocation).")

    rng = RNG_KEY
    params = None
    opt_state = None
    pad_idx = processor.vocab['<PAD>']
    current_len = 0

    # ======= Resume (generic) =======
    start_stage_idx = 0
    generic_latest = os.path.join(CHECKPOINT_DIR, 'latest.pkl')
    resume_payload = try_load_checkpoint(generic_latest)
    if resume_payload is not None and 'params' in resume_payload:
        ckpt_len = resume_payload.get('max_len', resume_payload['params']['encoder']['mps'].shape[0])
        for i, L in enumerate(LENGTH_STAGES):
            if L >= ckpt_len:
                start_stage_idx = i; break
        print(f"🔁 Resume: found generic checkpoint (len={ckpt_len}). Will start from stage #{start_stage_idx+1} (MAX_LEN={LENGTH_STAGES[start_stage_idx]}).")
    else:
        print("🔁 Resume: no generic checkpoint found. Starting from the first stage.")

    # ------- helper to (re)build optimizer & step -------
    def build_opt_and_step(steps_est: int, helper_symbols: int):
        opt = make_optimizer(steps_est)
        if USE_SYMBOLICS:
            # ensure symbolic params before opt.init
            nonlocal params
            params = ensure_symbolic_params(params, helper_symbols, rng)
        state = opt.init(params)
        if IS_MULTI:
            step_fn = make_train_step_symbolic_pmap(opt, axis_name=PMAP_AXIS_NAME)
        else:
            step_fn = make_train_step_symbolic(opt) if USE_SYMBOLICS else None
        return opt, state, step_fn

    for stage_idx in range(start_stage_idx, len(LENGTH_STAGES)):
        MAX_LEN = LENGTH_STAGES[stage_idx]

        # Stage batch: if multi-GPU, force multiple of N_DEV
        base_stage_batch = compute_stage_batch_size(MAX_LEN)
        if IS_MULTI:
            per_dev = max(1, base_stage_batch // N_DEV)
            STAGE_BATCH = per_dev * N_DEV
        else:
            STAGE_BATCH = base_stage_batch

        print("\n======================================================================")
        print(f"   STAGE {stage_idx+1}/{len(LENGTH_STAGES)} — MAX_LEN = {MAX_LEN} | STAGE_BATCH = {STAGE_BATCH}{' ('+str(N_DEV)+'x'+str(STAGE_BATCH//N_DEV)+')' if IS_MULTI else ''}")
        print("======================================================================")

        stage_selected = stages_data[stage_idx]['selected']

        if len(stage_selected) == 0:
            print("⚠️  No sequences for this stage. Skipping.")
            continue

        # ------- Recover per-stage data_state if compatible -------
        data_state = None
        if resume_payload is not None:
            data_state = resume_payload.get('data_state', None)

        # default fresh splits
        use_recovered = False
        if (data_state is not None
            and data_state.get('stage_idx', -1) == stage_idx
            and tuple(data_state.get('bounds', (-1,-1))) == tuple(bounds_list[stage_idx])):
            # Restore exact splits & hyper-state
            train_seqs = data_state['train_seqs']
            val_seqs   = data_state['val_seqs']
            test_seqs  = data_state['test_seqs']
            did_upsample_events = int(data_state.get('did_upsample_events', 0))
            last_upsample_epoch = int(data_state.get('last_upsample_epoch', -1))
            curr_dropout        = float(data_state.get('curr_dropout', DROPOUT_RATE_TRAIN))
            curr_smoothing      = float(data_state.get('curr_smoothing', LABEL_SMOOTHING))
            # RNG states (best-effort)
            rng_pack = resume_payload.get('rng_state', {})
            try:
                if 'py' in rng_pack: random.setstate(rng_pack['py'])
                if 'np' in rng_pack: np.random.set_state(rng_pack['np'])
                if 'jax_key' in rng_pack: rng = rng_pack['jax_key']
            except Exception:
                pass
            use_recovered = True
            print("🔁 Recovered data_state for this stage (train/val/test + upsampling hyper-state).")
        else:
            train_seqs, val_seqs, test_seqs = split_for_len(stage_selected, seed=42)
            did_upsample_events = 0
            last_upsample_epoch = -1
            curr_dropout   = DROPOUT_RATE_TRAIN
            curr_smoothing = LABEL_SMOOTHING

        # Build arrays
        train_arr = build_arrays(processor, train_seqs, MAX_LEN)
        val_arr   = build_arrays(processor, val_seqs,   MAX_LEN)
        test_arr  = build_arrays(processor, test_seqs,  MAX_LEN)

        # Deterministic stage_extra excluding ALL unique in train/val/test
        used_unique = {seq_key(s) for s in (train_seqs + val_seqs + test_seqs)}
        stage_extra = rebuild_stage_extra(sequences, bounds_list[stage_idx], used_unique)

        # ---- SYMBOLICS for TRAIN ONLY (memory safe)
        if USE_SYMBOLICS:
            helper = SymbolicsHelper(processor, config=DEFAULT_PROTEIN_CONFIG)
            train_masks_u8, train_symv_u8, train_neighbors_local = helper.build_for_subset(train_seqs, MAX_LEN)
            print(f"Symbolics subset ready: train={len(train_seqs)}  masks={train_masks_u8.shape}")
        else:
            train_masks_u8 = np.ones((len(train_seqs), MAX_LEN-1, VOCAB_SIZE), dtype=np.uint8)
            train_symv_u8  = np.zeros((len(train_seqs), 1), dtype=np.uint8)
            train_neighbors_local = np.full((len(train_seqs), 1), -1, dtype=np.int32)

        steps_per_epoch = max(1, (len(train_arr) // STAGE_BATCH))
        total_steps_est = steps_per_epoch * EPOCHS_PER_STAGE

        # Load or init params (length-aware)
        loaded, loaded_path = load_checkpoint_for_len(CHECKPOINT_DIR, MAX_LEN)
        if loaded is not None:
            params, opt_state = loaded.get('params'), loaded.get('opt_state')
            start_epoch = loaded.get('epoch', -1) + 1
            ckpt_len = params['encoder']['mps'].shape[0]
            if ckpt_len < MAX_LEN:
                print(f"ℹ️  The checkpoint ({loaded_path}) has len={ckpt_len}. Growing to len={MAX_LEN} …")
                params = grow_params(params, ckpt_len, MAX_LEN, BOND_DIMENSION_MPS, BOND_DIMENSION_MERA, VOCAB_SIZE, rng)
                set_pad_identity(params, pad_idx)
                start_epoch = 0
            elif ckpt_len > MAX_LEN:
                print(f"ℹ️  The checkpoint ({loaded_path}) has len={ckpt_len}. Shrinking to len={MAX_LEN} …")
                params = shrink_params(params, ckpt_len, MAX_LEN, BOND_DIMENSION_MPS, BOND_DIMENSION_MERA)
                set_pad_identity(params, pad_idx)
                start_epoch = 0
            else:
                set_pad_identity(params, pad_idx)
            print(f"✅ Loaded checkpoint from '{loaded_path}' (epoch {start_epoch})")
        else:
            if params is not None and current_len > 0 and current_len != MAX_LEN:
                prev_len = current_len
                print(f"➡️  Growing parameters from {prev_len} ➜ {MAX_LEN} …")
                params = grow_params(params, prev_len, MAX_LEN, BOND_DIMENSION_MPS, BOND_DIMENSION_MERA, VOCAB_SIZE, rng)
                set_pad_identity(params, pad_idx)
                start_epoch = 0
            elif params is None:
                print("Starting fresh training… (init params)")
                params = init_encoder_decoder_model(RNG_KEY, VOCAB_SIZE, BOND_DIMENSION_MPS, BOND_DIMENSION_MERA, MAX_LEN)
                set_pad_identity(params, pad_idx)
                start_epoch = 0
            else:
                start_epoch = 0

        current_len = MAX_LEN

        # Build optimizer & step (single or multi)
        optimizer, opt_state, train_step_sym = build_opt_and_step(total_steps_est, helper.num_symbols if USE_SYMBOLICS else 0)

        # Warm-up JIT (single device suffices)
        if len(train_arr) >= STAGE_BATCH:
            dummy = jnp.asarray(train_arr[:min(STAGE_BATCH, len(train_arr))])
            rng, subk = jax.random.split(rng)
            _ = loss_fn(params, dummy, dummy, pad_idx, subk, 0.0, LABEL_SMOOTHING).block_until_ready()

        # ---- Training loop (adaptive upsampling + HEM) ----
        best_val = float('inf')
        best_epoch = -1
        best_payload_mem = None      # buffered best (in-memory)
        last_best_saved_epoch = -1   # for periodic best saving
        no_improve = 0
        stage_t0 = time.time()
        if not use_recovered:
            did_upsample_events = 0
            last_upsample_epoch = -1
            curr_dropout = DROPOUT_RATE_TRAIN
            curr_smoothing = LABEL_SMOOTHING

        val_hist: List[float] = []
        train_hist: List[float] = []
#####################WARMUP TRIGGER GIA UPSAMPLING
        min_trigger_epoch = max(1, int(0.1 * MIN_EPOCHS))

        for epoch in range(start_epoch, EPOCHS_PER_STAGE):
            t0=time.time()
            perm = np.random.permutation(len(train_arr))
            total_loss=0.0; nb=0; total_tokens=0

            # small schedules for symbolics
            alpha_e = 0.5 if USE_SYMBOLICS else 0.0
            tau_e   = 0.9 if USE_SYMBOLICS else 1.0
            lambda_graph_e = LAMBDA_GRAPH_BASE * (1.0 + 0.3 * min(1.0, epoch/50.0)) if USE_SYMBOLICS else 0.0

            for i in range(0, len(train_arr), STAGE_BATCH):
                idx = perm[i:i+STAGE_BATCH]
                if len(idx) < STAGE_BATCH:
                    continue

                if IS_MULTI:
                    # Shard inputs per device
                    batch_np = np.asarray(train_arr[idx])
                    masks_np = np.asarray(train_masks_u8[idx])
                    symv_np  = np.asarray(train_symv_u8[idx])

                    batch_sh = shard_first_axis(batch_np, N_DEV)
                    masks_sh = shard_first_axis(masks_np, N_DEV)
                    symv_sh  = shard_first_axis(symv_np,  N_DEV)

                    # Build pairs per device (host-side)
                    per_dev_pairs = []
                    per_dev_masks = []
                    per = STAGE_BATCH // N_DEV
                    for d in range(N_DEV):
                        local_idx = idx[d*per:(d+1)*per]
                        p_pad, p_mask = get_pairs_for_batch(
                            local_idx, train_neighbors_local,
                            edges_per_node=GRAPH_EDGES_PER_NODE,
                            max_pairs=GRAPH_MAX_PAIRS,
                        )
                        per_dev_pairs.append(np.array(p_pad))
                        per_dev_masks.append(np.array(p_mask))
                    pairs_sh, pmask_sh = shard_pairs(per_dev_pairs, per_dev_masks)

                    # RNG keys per device
                    rng, subk = jax.random.split(rng)
                    dev_keys = jax.random.split(subk, N_DEV)

                    params, opt_state, loss = train_step_sym(
                        params, opt_state,
                        jnp.asarray(batch_sh),
                        int(pad_idx),
                        dev_keys,
                        float(curr_dropout), float(curr_smoothing),
                        jnp.asarray(masks_sh),
                        jnp.asarray(symv_sh),
                        jnp.asarray(pairs_sh),
                        jnp.asarray(pmask_sh),
                        float(lambda_graph_e), float(alpha_e), float(tau_e),
                    )
                else:
                    batch = jnp.asarray(train_arr[idx])
                    rng, subk = jax.random.split(rng)

                    if USE_SYMBOLICS:
                        pairs_padded, pairs_mask = get_pairs_for_batch(
                            idx, train_neighbors_local,
                            edges_per_node=GRAPH_EDGES_PER_NODE,
                            max_pairs=GRAPH_MAX_PAIRS,
                        )
                        params, opt_state, loss = train_step_sym(
                            params, opt_state,
                            batch_inputs=batch,
                            pad_idx=pad_idx, rng_key=subk,
                            dropout_rate=curr_dropout, label_smoothing=curr_smoothing,
                            allowed_masks_uint8=jnp.asarray(train_masks_u8[idx]),
                            symbol_vecs_uint8=jnp.asarray(train_symv_u8[idx]),
                            pairs_padded=pairs_padded, pairs_mask=pairs_mask,
                            lambda_graph=lambda_graph_e, alpha=alpha_e, tau=tau_e,
                        )
                    else:
                        l, grads = jax.value_and_grad(loss_fn)(
                            params, batch, batch, pad_idx, subk, curr_dropout, curr_smoothing
                        )
                        updates, opt_state = optimizer.update(grads, opt_state, params)
                        params = optax.apply_updates(params, updates)
                        loss = l

                total_loss += float(loss); nb += 1
                total_tokens += int((np.asarray(train_arr[idx])[:,1:] != pad_idx).sum())

            train_ppl = float(np.exp(total_loss/nb)) if nb>0 else float('inf')
            val_ppl   = compute_perplexity(
                params, val_arr, pad_idx,
                batch=min(STAGE_BATCH, len(val_arr)),
                label_smoothing=0.0
            )
            dt = time.time()-t0
            print(f"[len={MAX_LEN:>3}, bs={STAGE_BATCH:>3}{' ('+str(N_DEV)+'x'+str(STAGE_BATCH//N_DEV)+')' if IS_MULTI else ''}] "
                  f"Epoch {epoch+1:>3}/{EPOCHS_PER_STAGE} | {dt:.1f}s | train-PPL {train_ppl:.3f} | val-PPL {val_ppl:.3f} | tokens {total_tokens}")

            train_hist.append(train_ppl); val_hist.append(val_ppl)

            # ---------- BEST (buffered / periodic) ----------
            improved = val_ppl + MIN_DELTA < best_val
            if improved:
                best_val = val_ppl
                best_epoch = epoch
                best_payload_mem = {
                    'params': params,
                    'opt_state': opt_state,
                    'epoch': epoch,
                    'max_len': current_len,
                    'data_state': {
                        'stage_idx': stage_idx,
                        'bounds': bounds_list[stage_idx],
                        'length_stages': LENGTH_STAGES,
                        'train_seqs': train_seqs,
                        'val_seqs':   val_seqs,
                        'test_seqs':  test_seqs,
                        'did_upsample_events': did_upsample_events,
                        'last_upsample_epoch': last_upsample_epoch,
                        'curr_dropout':   curr_dropout,
                        'curr_smoothing': curr_smoothing,
                    },
                    'rng_state': {
                        'py': random.getstate(),
                        'np': np.random.get_state(),
                        'jax_key': rng,
                    }
                }
                # Periodic write according to SAVE_BEST_EVERY
                should_write_best_now = (
                    SAVE_BEST and
                    ((SAVE_BEST_EVERY == 1) or (SAVE_BEST_EVERY > 1 and ((epoch + 1) % SAVE_BEST_EVERY == 0)))
                )
                if should_write_best_now:
                    try:
                        save_checkpoint_dual(best_payload_mem, CHECKPOINT_DIR, MAX_LEN, is_best=True)
                        last_best_saved_epoch = epoch + 1
                    except Exception as e:
                        print("❌ Failed to save best checkpoint (periodic):", e)
                else:
                    print(f"⭐ New best buffered (val-PPL {best_val:.3f}) — will flush later.")
                no_improve = 0
            else:
                no_improve += 1

            # ---------- PERIODIC 'LATEST' SAVE ----------
            if SAVE_LATEST_EVERY and SAVE_LATEST_EVERY > 0 and ((epoch + 1) % SAVE_LATEST_EVERY == 0):
                try:
                    payload = {
                        'params': params,
                        'opt_state': opt_state,
                        'epoch': epoch,
                        'max_len': current_len,
                        'data_state': {
                            'stage_idx': stage_idx,
                            'bounds': bounds_list[stage_idx],
                            'length_stages': LENGTH_STAGES,
                            'train_seqs': train_seqs,
                            'val_seqs':   val_seqs,
                            'test_seqs':  test_seqs,
                            'did_upsample_events': did_upsample_events,
                            'last_upsample_epoch': last_upsample_epoch,
                            'curr_dropout':   curr_dropout,
                            'curr_smoothing': curr_smoothing,
                        },
                        'rng_state': {
                            'py': random.getstate(),
                            'np': np.random.get_state(),
                            'jax_key': rng,
                        }
                    }
                    save_checkpoint_dual(payload, CHECKPOINT_DIR, MAX_LEN, is_best=False)
                except Exception as e:
                    print("❌ Failed to save periodic checkpoint:", e)

            # ---- Adaptive upsampling trigger ----
            try_upsample, reason = should_adaptive_upsample(
                epoch_idx=epoch, val_hist=val_hist, best_val=best_val,
                last_upsample_epoch=last_upsample_epoch,
                min_trigger_epoch=min_trigger_epoch,
                plateau_patience=UPSAMPLE_TRIGGER_PATIENCE,
                window=REL_IMPROVEMENT_WINDOW,
                min_rel_improve=MIN_REL_IMPROVEMENT,
                abs_ceiling=PPL_ABS_CEILING,
                cooldown_epochs=COOLDOWN_EPOCHS,
                min_required=MIN_VAL_PPL_REQUIRED,
            )

            if ADAPTIVE_UPSAMPLING and try_upsample and did_upsample_events < MAX_UPSAMPLE_EVENTS_PER_STAGE:
                old_n = len(train_seqs)
                desired_factor = min(MAX_UPSAMPLE_FACTOR, UPSAMPLE_FACTOR_BASE + UPSAMPLE_FACTOR_STEP * did_upsample_events)
                strategy = 'hard' if (ENABLE_HEM and HEM_ON_UPSAMPLE) else 'random'
                new_train_seqs, stage_extra, stats = upsample_train_set_general(
                    base_train=train_seqs,
                    extra_pool=stage_extra,
                    factor=desired_factor,
                    repeat_cap_multiplier=REPEAT_CAP_MULTIPLIER,
                    seed=PY_SEED + 777 + stage_idx*1000 + epoch,
                    strategy=strategy,
                    params=params, processor=processor, max_len=MAX_LEN, pad_idx=pad_idx,
                    curr_smoothing=curr_smoothing
                )
                if len(new_train_seqs) > old_n:
                    train_seqs = new_train_seqs
                    train_arr = build_arrays(processor, train_seqs, MAX_LEN)

                    # >>> REBUILD SYMBOLIC BUNDLES for NEW TRAIN SET <<<
                    if USE_SYMBOLICS:
                        helper = SymbolicsHelper(processor, config=DEFAULT_PROTEIN_CONFIG)
                        train_masks_u8, train_symv_u8, train_neighbors_local = helper.build_for_subset(train_seqs, MAX_LEN)
                    # <<< /REBUILD >>>

                    # Recompute steps & rebuild optimizer/step (reset opt state)
                    if IS_MULTI:
                        per_dev = max(1, compute_stage_batch_size(MAX_LEN) // N_DEV)
                        STAGE_BATCH = per_dev * N_DEV
                    else:
                        STAGE_BATCH = compute_stage_batch_size(MAX_LEN)

                    steps_per_epoch = max(1, (len(train_arr) // STAGE_BATCH))
                    total_steps_est = steps_per_epoch * max(1, (EPOCHS_PER_STAGE - epoch - 1))
                    optimizer, opt_state, train_step_sym = build_opt_and_step(total_steps_est, helper.num_symbols if USE_SYMBOLICS else 0)

                    curr_dropout = UPSAMPLED_DROPOUT
                    curr_smoothing = UPSAMPLED_SMOOTH
                    did_upsample_events += 1
                    last_upsample_epoch = epoch + 1
                    print(f"🟩 Upsample[{strategy}] {reason} at epoch {epoch+1}: "
                          f"train {old_n} → {len(train_seqs)} "
                          f"(+uniq {stats['added_unique']}, +rep {stats['added_repeats']}), "
                          f"dropout→{curr_dropout}, smoothing→{curr_smoothing}")
                    no_improve = 0
                else:
                    print(f"ℹ️ Upsample skipped (no capacity). Reason={reason}")

            # Early stop per stage (respect MIN_VAL_PPL_REQUIRED)
            allow_stop = (MIN_VAL_PPL_REQUIRED is None) or (best_val <= MIN_VAL_PPL_REQUIRED)
            if (epoch + 1) >= MIN_EPOCHS and no_improve >= PATIENCE:
                if allow_stop:
                    print(f"🟨 Plateau at len={MAX_LEN} (no improvement {PATIENCE}x after {MIN_EPOCHS}+ epochs, best val-PPL {best_val:.3f}). "
                          f"Stopping stage (threshold ok: {MIN_VAL_PPL_REQUIRED}).")
                    # Flush buffered BEST (if any)
                    if FLUSH_BEST_ON_EARLY_STOP and best_payload_mem is not None and SAVE_BEST:
                        try:
                            save_checkpoint_dual(best_payload_mem, CHECKPOINT_DIR, MAX_LEN, is_best=True)
                            last_best_saved_epoch = epoch + 1
                        except Exception as e:
                            print("❌ Failed to save BEST on early stop:", e)
                    # Force-save LATEST on early stop
                    if SAVE_LATEST_ON_EARLY_STOP:
                        try:
                            payload = {
                                'params': params,
                                'opt_state': opt_state,
                                'epoch': epoch,
                                'max_len': current_len,
                                'data_state': {
                                    'stage_idx': stage_idx,
                                    'bounds': bounds_list[stage_idx],
                                    'length_stages': LENGTH_STAGES,
                                    'train_seqs': train_seqs,
                                    'val_seqs':   val_seqs,
                                    'test_seqs':  test_seqs,
                                    'did_upsample_events': did_upsample_events,
                                    'last_upsample_epoch': last_upsample_epoch,
                                    'curr_dropout':   curr_dropout,
                                    'curr_smoothing': curr_smoothing,
                                },
                                'rng_state': {
                                    'py': random.getstate(),
                                    'np': np.random.get_state(),
                                    'jax_key': rng,
                                }
                            }
                            save_checkpoint_dual(payload, CHECKPOINT_DIR, MAX_LEN, is_best=False)
                        except Exception as e:
                            print("❌ Failed to save checkpoint on early stop:", e)
                    break
                else:
                    print(f"🟪 Early-stop deferred: best val-PPL {best_val:.3f} > required {MIN_VAL_PPL_REQUIRED:.3f}. "
                          f"Continuing training & upsampling as needed.")
                    no_improve = 0  # avoid spamming

        stage_dt = time.time()-stage_t0
        print(f"⏱️  Stage time (len={MAX_LEN}): {stage_dt:.1f}s")

        # Final saves at stage end
        # Flush buffered BEST (if any)
        if FLUSH_BEST_ON_STAGE_END and best_payload_mem is not None and SAVE_BEST:
            try:
                save_checkpoint_dual(best_payload_mem, CHECKPOINT_DIR, MAX_LEN, is_best=True)
                last_best_saved_epoch = best_payload_mem['epoch'] + 1
            except Exception as e:
                print("❌ Failed to save BEST at stage end:", e)

        # Force-save LATEST at stage end
        if SAVE_LATEST_ON_STAGE_END:
            try:
                payload = {
                    'params': params,
                    'opt_state': opt_state,
                    'epoch': epoch,  # last epoch run
                    'max_len': current_len,
                    'data_state': {
                        'stage_idx': stage_idx,
                        'bounds': bounds_list[stage_idx],
                        'length_stages': LENGTH_STAGES,
                        'train_seqs': train_seqs,
                        'val_seqs':   val_seqs,
                        'test_seqs':  test_seqs,
                        'did_upsample_events': did_upsample_events,
                        'last_upsample_epoch': last_upsample_epoch,
                        'curr_dropout':   curr_dropout,
                        'curr_smoothing': curr_smoothing,
                    },
                    'rng_state': {
                        'py': random.getstate(),
                        'np': np.random.get_state(),
                        'jax_key': rng,
                    }
                }
                save_checkpoint_dual(payload, CHECKPOINT_DIR, MAX_LEN, is_best=False)
            except Exception as e:
                print("❌ Failed to save checkpoint at stage end:", e)

        # Stage Test (use in-memory best if present)
        if best_payload_mem is not None:
            params = best_payload_mem['params']
        test_ppl = compute_perplexity(
            params, test_arr, pad_idx,
            batch=min(STAGE_BATCH, len(test_arr)),
            label_smoothing=0.0
        )
        print(f"✅ [len={MAX_LEN}] TEST Perplexity: {test_ppl:.3f}")

    # ---- SAVE FINAL MODEL ----
    try:
        atomic_pickle_save((params, processor, current_len), MODEL_FILE)
        size_mb = os.path.getsize(MODEL_FILE)/1024/1024
        print(f"\n✅ Saved: {MODEL_FILE} ({size_mb:.2f} MB)")
    except Exception as e:
        print("❌ Failed to save model:", e)

    total_dt = time.time()-start_global
    print(f"🏁 Curriculum complete. Total time: {total_dt:.1f}s")

# ===================== Entry =====================

if __name__ == '__main__':
    start_global = time.time()
    main()


# Run training

In [ ]:
!python /kaggle/working/train_with_symbolics.py

# Fine tuning

In [ ]:
%%writefile /kaggle/working/range_finetune_master.py
# -*- coding: utf-8 -*-
# ==============================================================================
#   MERA-MPS range-aware fine-tuning (AMP-ready)
#   - Loads base LM from production_model_bio.pkl
#   - Filters Pfam by length window; optional upsample near target length
#   - Two phases: A) freeze encoder, B) unfreeze all
#   - Grows/shrinks to TARGET_LEN; saves tuned model (production_model_tuned.pkl)
#   - Deterministic-ish, memory-safe (JAX preallocate OFF)
# ==============================================================================

import os, re, time, math, random, pickle, warnings
from typing import List, Tuple, Optional

# ---- JAX memory safety (must be set BEFORE importing jax) ----
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.80")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "default")

import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
import jax.tree_util as jtu
import optax
warnings.filterwarnings("ignore", category=UserWarning)

# ========================== USER SETTINGS (tuned for AMP) ====================
BASE_MODEL   = os.environ.get("BASE_MODEL", "/kaggle/input/model1/other/default/1/production_model_bio.pkl")
TUNED_MODEL  = os.environ.get("TUNED_MODEL", "/kaggle/working/production_model_tuned.pkl")

PFAM_FILES   = [
    "/kaggle/input/pfaseed/Pfam-A.seed",
    "/kaggle/input/pfafull/Pfam-A.full",   # αν λείπει, απλά προχωράμε
]

# ---- Εύρος για AMPs (συνήθως 24–50 aa). Ρύθμισε αν θες πιο φαρδύ range.
KEEP_WINDOW: Tuple[int,int] = (24, 52)    # inclusive

# Αν None → αυτόματο mid-even. Για AMPs ένα καλό anchor είναι 40.
TARGET_LEN: Optional[int] = 40

# Upsampling γύρω από το anchor length (σταθεροποιεί το tokenizer και το MERA)
NEAR_DELTA        = 2
NEAR_TARGET_MIN   = 1800   # αυξάνει βήματα/χρόνο, αλλά δίνει πιο σταθερό μοντέλο

# Batch sizing
TOKENS_PER_BATCH  = 8192
BATCH_MAX         = 64

# Reproducibility
PY_SEED           = 0

# Phase A (freeze encoder)
EPOCHS_A   = 12
LR_A       = 1.0e-4
DROPOUT_A  = 0.10
SMOOTH_A   = 0.02
PATIENCE_A = 4

# Phase B (unfreeze all)
EPOCHS_B   = 120
LR_B       = 8.0e-5
DROPOUT_B  = 0.10
SMOOTH_B   = 0.02
PATIENCE_B = 8

# Optional capacity bump (None = keep as-is)
BOND_DIM_MPS_TUNED: Optional[int]  = None   # e.g. 48
BOND_DIM_MERA_TUNED: Optional[int] = None   # e.g. 48

# ========================= FORMAT CHECK ======================================
def is_compact(params):
    try:
        return ('encoder' in params and isinstance(params['encoder'], dict)
                and 'mps' in params['encoder'] and 'isos' in params['encoder'])
    except Exception:
        return False

# ========================= DATA HELPERS ======================================
VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

def _is_seq_line(line: str) -> bool:
    if (not line) or line.startswith('#') or line.startswith('//'):
        return False
    parts = line.split()
    if len(parts) < 2:
        return False
    seq = parts[-1]
    return bool(re.fullmatch(r'[A-Za-z\-\.]+', seq))

def read_pfam_sequences(file_paths: List[str], max_len_cap: int=256) -> List[List[str]]:
    seqs = []
    for path in file_paths:
        if not path or not os.path.exists(path):
            print(f"⚠️  Missing file: {path}")
            continue
        try:
            with open(path, 'r', encoding='utf-8') as f:
                for raw in f:
                    line = raw.strip()
                    if not _is_seq_line(line):
                        continue
                    s = line.split()[-1].upper()
                    s = ''.join([c for c in s if c in VALID_AA])
                    if len(s) >= 1:
                        s = s[:max_len_cap-2]   # -2 for SOS/EOS
                        seqs.append(list(s))
        except Exception as e:
            print(f"⚠️  Failed to read {path}: {e}")
    # dedupe
    uniq, seen = [], set()
    for lst in seqs:
        ss = ''.join(lst)
        if ss not in seen:
            seen.add(ss); uniq.append(lst)
    return uniq

def length_filter_and_upsample(uniq: List[List[str]],
                               keep_window: Tuple[int,int],
                               target_len: int,
                               near_delta: int,
                               near_target_min: int,
                               seed: int = 123) -> List[List[str]]:
    lo, hi = keep_window
    uniq = [s for s in uniq if lo <= len(s) <= hi]
    # upsample near target_len±delta
    near = [s for s in uniq if target_len-near_delta <= len(s) <= target_len+near_delta]
    rng = random.Random(seed)
    added = 0
    base = list(uniq)
    if len(near) > 0 and len(near) < near_target_min:
        need = near_target_min - len(near)
        base += [rng.choice(near) for _ in range(need)]
        added = need
    print(f"Upsampled near {target_len}±{near_delta}: +{added} (final {len(base)})")
    return base

# ========================= MODEL/PACKING SHIMS ===============================
class DataProcessor:
    """Minimal shim: αρκεί για unpickle + vocab + packing."""
    def sequence_to_indices(self, seq):
        vocab = getattr(self, "vocab", {}) or {}
        unk = vocab.get("<UNK>", 0)
        return np.asarray([vocab.get(tok, unk) for tok in seq], dtype=np.int32)

def pad_and_pack(processor, seq_chars: List[str], max_len: int):
    sos=processor.vocab['<SOS>']; eos=processor.vocab['<EOS>']; pad=processor.vocab['<PAD>']
    seq = ['<SOS>'] + list(seq_chars)[:max_len-2] + ['<EOS>']
    idx = processor.sequence_to_indices(seq)
    if len(idx) < max_len:
        idx = np.concatenate([idx, np.full((max_len-len(idx),), pad, np.int32)])
    return idx

def build_arrays(processor, seq_lists: List[List[str]], max_len: int):
    if len(seq_lists) == 0:
        return np.empty((0, max_len), dtype=np.int32)
    return np.asarray([pad_and_pack(processor, s, max_len) for s in seq_lists], dtype=np.int32)

def set_pad_identity(params, pad_idx: int):
    mps_enc = params['encoder']['mps']
    mps_dec = params['decoder']['mps']
    L, V, chi, _ = mps_enc.shape
    I = jnp.eye(chi, dtype=mps_enc.dtype)
    params['encoder']['mps'] = mps_enc.at[:, pad_idx].set(I)
    params['decoder']['mps'] = mps_dec.at[:, pad_idx].set(I)

def grow_params(params, old_len, new_len, bond_dim_mps, bond_dim_mera, vocab_size, rng):
    assert new_len > old_len and new_len % 2 == 0
    def grow_block(block, rng_block):
        mps_old = block['mps']; isos_old = block['isos']
        add_L = new_len - old_len
        k1, k2 = jax.random.split(rng_block)
        mps_new = jnp.zeros((new_len, vocab_size, bond_dim_mps, bond_dim_mps), dtype=mps_old.dtype)
        mps_new = mps_new.at[:old_len].set(mps_old * 0.999)
        if add_L > 0:
            init_tail = jax.random.normal(k1, (add_L, vocab_size, bond_dim_mps, bond_dim_mps)) * 0.1
            mps_new = mps_new.at[old_len:].set(init_tail)
        old_d = old_len // 2; new_d = new_len // 2
        isos_new = jnp.zeros((new_d, bond_dim_mps, bond_dim_mps, bond_dim_mera), dtype=isos_old.dtype)
        isos_new = isos_new.at[:old_d].set(isos_old * 0.999)
        if new_d - old_d > 0:
            init_iso_tail = jax.random.normal(k2, (new_d - old_d, bond_dim_mps, bond_dim_mps, bond_dim_mera)) * 0.1
            isos_new = isos_new.at[old_d:].set(init_iso_tail)
        return {'mps': mps_new, 'isos': isos_new}
    rng_enc, rng_dec, kproj = jax.random.split(rng, 3)
    new_encoder = grow_block(params['encoder'], rng_enc)
    new_decoder = grow_block(params['decoder'], rng_dec)
    old_td = (old_len//2) * bond_dim_mera
    new_td = (new_len//2) * bond_dim_mera
    proj_old = params['projection_matrix']
    proj_new = jnp.zeros((new_td, bond_dim_mps), dtype=proj_old.dtype)
    k = min(old_td, new_td)
    proj_new = proj_new.at[:k, :].set(proj_old[:k, :])
    if new_td > k:
        init_tail = jax.random.normal(kproj, (new_td - k, bond_dim_mps)) * 0.1
        proj_new = proj_new.at[k:, :].set(init_tail)
    return {
        'encoder': new_encoder,
        'decoder': new_decoder,
        'projection_matrix': proj_new,
        'output_projection': params['output_projection'],
    }

def shrink_params(params, old_len, new_len):
    assert new_len < old_len and new_len % 2 == 0
    def shrink_block(block):
        mps_old = block['mps']; isos_old = block['isos']
        return {'mps': mps_old[:new_len], 'isos': isos_old[:(new_len//2)]}
    new_encoder = shrink_block(params['encoder'])
    new_decoder = shrink_block(params['decoder'])
    old_td = (old_len//2) * params['encoder']['isos'].shape[-1]
    new_td = (new_len//2) * new_encoder['isos'].shape[-1]
    proj_old = params['projection_matrix']
    proj_new = proj_old[:new_td]
    return {
        'encoder': new_encoder,
        'decoder': new_decoder,
        'projection_matrix': proj_new,
        'output_projection': params['output_projection'],
    }

# ========================= ENCODE/DECODE (compact) ===========================
def _encode_step(carry, token_and_pos):
    state, mps, pos, rng_key, rate = carry
    tok, p = token_and_pos
    W = mps[p, tok]
    state = state @ W
    key = jax.random.fold_in(rng_key, p)
    keep = 1.0 - rate
    mask = jax.random.bernoulli(key, p=jnp.clip(keep, 0.0, 1.0), shape=state.shape)
    state = jnp.where(mask, state / jnp.maximum(keep, 1e-6), 0.0)
    return (state, mps, p+1, rng_key, rate), state

@jax.jit
def encode_compact(enc_params, seq_indices, rng_key, dropout_rate):
    L = enc_params['mps'].shape[0]
    chi = enc_params['mps'].shape[-1]
    state0 = jnp.zeros((chi,), dtype=jnp.float32).at[0].set(1.0)
    positions = jnp.arange(L, dtype=jnp.int32)
    (_, _, _, _, _), states = lax.scan(
        _encode_step,
        (state0, enc_params['mps'], 0, rng_key, dropout_rate),
        (seq_indices, positions)
    )
    def pair_reduce(_, i):
        l = states[2*i]; r = states[2*i+1]
        iso = enc_params['isos'][i]
        hi  = jnp.einsum('i,j,ijk->k', l, r, iso)
        return None, hi
    _, hi_list = lax.scan(pair_reduce, None, jnp.arange(L//2))
    tv = hi_list.reshape(-1)
    return tv / (jnp.linalg.norm(tv) + 1e-9)

def _decode_step(carry, tok_and_pos):
    vec, ctx, mps_dec, out_proj, first_flag = carry
    tok, p = tok_and_pos
    W = mps_dec[p, tok]
    vec = vec @ W
    vec = jnp.where(first_flag, vec + ctx, vec)
    logits = vec @ out_proj
    return (vec, ctx, mps_dec, out_proj, False), logits

@jax.jit
def decode_tf_compact(params, thought_vector, decoder_input):
    L = params['decoder']['mps'].shape[0]
    chi = params['decoder']['mps'].shape[-1]
    ctx = thought_vector @ params['projection_matrix']
    vec0 = jnp.zeros((chi,), dtype=jnp.float32).at[0].set(1.0)
    positions = jnp.arange(L-1, dtype=jnp.int32)
    mps_dec = params['decoder']['mps'][:-1]
    (_, _, _, _, _), logits = lax.scan(
        _decode_step,
        (vec0, ctx, mps_dec, params['output_projection'], True),
        (decoder_input, positions)
    )
    return logits  # (L-1, V)

# ========================= LOSS / TRAIN ======================================
def _xent_logits_with_int_labels(logits, labels_int, label_smoothing=0.0):
    V = logits.shape[-1]
    ls = jnp.asarray(label_smoothing, dtype=logits.dtype)
    onehot = jax.nn.one_hot(labels_int, V)
    smoothed = optax.smooth_labels(onehot, ls)
    return optax.softmax_cross_entropy(logits=logits, labels=smoothed)

@jax.jit
def per_seq_losses(params, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing):
    B = batch_inputs.shape[0]
    enc_keys = jax.random.split(rng_key, B)
    tvs = jax.vmap(encode_compact, in_axes=(None,0,0,None))(
        params['encoder'], batch_inputs, enc_keys, dropout_rate
    )
    dec_in  = batch_inputs[:, :-1]
    dec_tgt = batch_inputs[:, 1:]
    logits  = jax.vmap(decode_tf_compact, in_axes=(None,0,0))(
        params, tvs, dec_in
    )  # (B,L-1,V)
    token_losses = _xent_logits_with_int_labels(logits, dec_tgt, label_smoothing)
    mask = (dec_tgt != pad_idx)
    per_seq_sum = (token_losses * mask).sum(axis=1)
    per_seq_cnt = mask.sum(axis=1)
    return per_seq_sum / jnp.maximum(per_seq_cnt, 1)

@jax.jit
def loss_fn(params, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing):
    losses = per_seq_losses(params, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing)
    counts = (batch_inputs[:, 1:] != pad_idx).sum(axis=1).astype(jnp.float32)
    denom = jnp.maximum(counts.sum(), 1.0)
    return (losses * counts).sum() / denom

def compute_batch_size(max_len: int) -> int:
    return max(8, min(BATCH_MAX, max(1, TOKENS_PER_BATCH // max(1, max_len))))

def make_optimizer(lr: float):
    return optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=lr, weight_decay=2e-4)
    )

def compute_perplexity(params, data_arr, pad_idx, batch=64):
    n = len(data_arr)
    if n == 0:
        return float('inf')
    batch = max(1, min(batch, n))
    rng = jax.random.PRNGKey(999)
    tot_ll = 0.0; tot_tokens = 0
    for i in range(0, n, batch):
        chunk_np = np.asarray(data_arr[i:i+batch])
        chunk = jnp.asarray(chunk_np)
        l = float(loss_fn(params, chunk, pad_idx, rng, 0.0, 0.0))
        nonpad = int((chunk_np[:, 1:] != pad_idx).sum())
        tot_ll += l * nonpad
        tot_tokens += nonpad
    if tot_tokens == 0:
        return float('inf')
    return float(np.exp(tot_ll / tot_tokens))

# ---------- train steps ----------
def make_train_step_freeze_encoder(pad_idx, optimizer):
    @jax.jit
    def step(params, opt_state, batch, rng_key, dropout_rate, ls):
        l, grads = jax.value_and_grad(loss_fn)(
            params, batch, pad_idx, rng_key, dropout_rate, ls
        )
        grads = dict(grads)
        grads['encoder'] = jtu.tree_map(jnp.zeros_like, grads['encoder'])
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, l
    return step

def make_train_step_unfreeze(pad_idx, optimizer):
    @jax.jit
    def step(params, opt_state, batch, rng_key, dropout_rate, ls):
        l, grads = jax.value_and_grad(loss_fn)(
            params, batch, pad_idx, rng_key, dropout_rate, ls
        )
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, l
    return step

# ========================= MAIN =============================================
def _evenize(n: int) -> int:
    return n if (n % 2 == 0) else (n - 1 if n > 1 else 2)

def _auto_target_from_window(win: Tuple[int,int]) -> int:
    mid = int(round(0.5*(win[0] + win[1])))
    return max(2, _evenize(mid))

def main():
    random.seed(PY_SEED); np.random.seed(PY_SEED)

    window = KEEP_WINDOW
    if window[0] < 1 or window[1] < window[0]:
        raise ValueError("Invalid KEEP_WINDOW.")
    tgt_len = TARGET_LEN if TARGET_LEN is not None else _auto_target_from_window(window)
    if tgt_len % 2 != 0:
        tgt_len = _evenize(tgt_len)
    if not (window[0] <= tgt_len <= window[1]):
        print(f"ℹ️ TARGET_LEN ({tgt_len}) not inside KEEP_WINDOW {window}. Clamping.")
        tgt_len = max(window[0], min(window[1], tgt_len))
        tgt_len = _evenize(tgt_len)

    if not os.path.exists(BASE_MODEL):
        raise FileNotFoundError(f"Base model not found: {BASE_MODEL}")
    with open(BASE_MODEL, "rb") as f:
        params, processor, model_len = pickle.load(f)
    if not is_compact(params):
        raise RuntimeError("Only compact param format supported.")

    chi_mps = int(params['encoder']['mps'].shape[-1])
    chim    = int(params['encoder']['isos'].shape[-1])
    vocab   = int(params['encoder']['mps'].shape[1])
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] ▶ Loaded base ✓  len={model_len}  χ={chi_mps}  chim={chim}  vocab={vocab}")
    print(f"Range window = {window} | TARGET_LEN = {tgt_len}")

    # optional capacity bump
    if BOND_DIM_MPS_TUNED is not None or BOND_DIM_MERA_TUNED is not None:
        def maybe_bump_capacity(params, bond_dim_mps_new=None, bond_dim_mera_new=None, rng=None):
            mps = params['encoder']['mps']; isos = params['encoder']['isos']
            L, V, chi_old, _ = mps.shape
            chim_old = isos.shape[-1]
            chi_new  = bond_dim_mps_new  if bond_dim_mps_new  is not None else chi_old
            chim_new = bond_dim_mera_new if bond_dim_mera_new is not None else chim_old
            if chi_new == chi_old and chim_new == chim_old:
                return params
            def grow_block(block, k1, k2):
                mps_old = block['mps']; isos_old = block['isos']
                L, V, chi_old, _ = mps_old.shape
                chim_old = isos_old.shape[-1]
                mps_new = jnp.zeros((L, V, chi_new, chi_new), dtype=mps_old.dtype)
                isos_new = jnp.zeros((L//2, chi_new, chi_new, chim_new), dtype=isos_old.dtype)
                mps_new = mps_new.at[..., :chi_old, :chi_old].set(mps_old)
                isos_new = isos_new.at[..., :chi_old, :chi_old, :chim_old].set(isos_old)
                if chi_new > chi_old:
                    init_tail = jax.random.normal(k1, (L, V, chi_new-chi_old, chi_new-chi_old)) * 0.05
                    mps_new = mps_new.at[..., chi_old:, chi_old:].set(init_tail)
                if chim_new > chim_old:
                    init_iso_tail = jax.random.normal(k2, (L//2, chi_new, chi_new, chim_new-chim_old)) * 0.05
                    isos_new = isos_new.at[..., :, :, chim_old:].set(init_iso_tail)
                return {'mps': mps_new, 'isos': isos_new}
            k1, k2 = jax.random.split(jax.random.PRNGKey(42))
            new_encoder = grow_block(params['encoder'], k1, k2)
            new_decoder = grow_block(params['decoder'], k1, k2)
            old_td = (new_encoder['mps'].shape[0]//2) * chim
            new_td = (new_encoder['mps'].shape[0]//2) * new_encoder['isos'].shape[-1]
            proj_old = params['projection_matrix']
            proj_new = jnp.zeros((new_td, new_encoder['mps'].shape[-1]), dtype=proj_old.dtype)
            kk = min(old_td, new_td); cc = min(proj_old.shape[1], new_encoder['mps'].shape[-1])
            proj_new = proj_new.at[:kk, :cc].set(proj_old[:kk, :cc])
            out_old = params['output_projection']
            out_new = jnp.zeros((new_encoder['mps'].shape[-1], out_old.shape[1]), dtype=out_old.dtype).at[:cc,:].set(out_old[:cc,:])
            return {'encoder': new_encoder, 'decoder': new_decoder,
                    'projection_matrix': proj_new, 'output_projection': out_new}
        params = maybe_bump_capacity(params, BOND_DIM_MPS_TUNED, BOND_DIM_MERA_TUNED, jax.random.PRNGKey(42))
        chi_mps = int(params['encoder']['mps'].shape[-1])
        chim    = int(params['encoder']['isos'].shape[-1])

    # build candidates
    cands = read_pfam_sequences(PFAM_FILES, max_len_cap=256)
    print(f"Raw candidates: {len(cands)} — filtering to window {window} and upsampling near L={tgt_len}…")
    uniq = length_filter_and_upsample(
        cands, keep_window=window, target_len=tgt_len,
        near_delta=NEAR_DELTA, near_target_min=NEAR_TARGET_MIN, seed=123
    )

    # prepare arrays @ tgt_len
    if model_len != tgt_len:
        if model_len > tgt_len:
            print(f"Shrinking model parameters: {model_len} → {tgt_len}")
            params = shrink_params(params, model_len, tgt_len)
        else:
            print(f"Growing model parameters: {model_len} → {tgt_len}")
            params = grow_params(params, model_len, tgt_len, chi_mps, chim, vocab, jax.random.PRNGKey(99))
        model_len = tgt_len

    pad_idx = processor.vocab['<PAD>']
    set_pad_identity(params, pad_idx)

    rng = random.Random(1337)
    rng.shuffle(uniq)
    n_total = len(uniq)
    if n_total == 0:
        print("⚠️ No data after filters. Consider widening KEEP_WINDOW.")
        return
    n_val = max(200, int(0.2 * n_total))
    train_seqs = uniq[:-n_val]
    val_seqs   = uniq[-n_val:]

    train_arr = build_arrays(processor, train_seqs, model_len)
    val_arr   = build_arrays(processor, val_seqs,   model_len)
    bs = compute_batch_size(model_len)
    print(f"Filtered unique={n_total} | train={len(train_arr)} val={len(val_arr)} | batch≈{bs}")

    # warm-up JIT
    if len(train_arr) >= 1:
        dummy = jnp.asarray(train_arr[:max(1, min(8, len(train_arr)))])
        _ = loss_fn(params, dummy, pad_idx, jax.random.PRNGKey(7), 0.0, 0.0).block_until_ready()

    # -------- Phase A: freeze encoder --------
    optimizer_a   = make_optimizer(LR_A)
    opt_state     = optimizer_a.init(params)
    train_step_A  = make_train_step_freeze_encoder(pad_idx, optimizer_a)
    best_val = float('inf'); best_params = params; no_imp = 0
    rng_key = jax.random.PRNGKey(0)

    for epoch in range(1, EPOCHS_A+1):
        t0 = time.time()
        perm = np.random.permutation(len(train_arr))
        tot = 0.0; nb=0; total_tokens=0
        for i in range(0, len(train_arr), bs):
            idx = perm[i:i+bs]
            if len(idx) == 0: continue
            batch = jnp.asarray(train_arr[idx])
            rng_key, subk = jax.random.split(rng_key)
            params, opt_state, loss = train_step_A(params, opt_state, batch, subk, DROPOUT_A, SMOOTH_A)
            tot += float(loss); nb += 1
            total_tokens += int((np.asarray(train_arr[idx])[:,1:] != pad_idx).sum())
        train_ppl = float(np.exp(tot/nb)) if nb>0 else float('inf')
        val_ppl = compute_perplexity(params, val_arr, pad_idx, batch=min(bs, len(val_arr)))
        dt = time.time()-t0
        print(f"[A] {epoch:03d}/{EPOCHS_A} | train {train_ppl:.4f} | val {val_ppl:.4f} | {dt:.1f}s")
        if val_ppl + 1e-3 < best_val:
            best_val = val_ppl; best_params = params; no_imp = 0
        else:
            no_imp += 1
            if no_imp >= PATIENCE_A:
                print("Early stop Phase A.")
                break
    params = best_params

    # -------- Phase B: unfreeze all --------
    optimizer_b   = make_optimizer(LR_B)
    opt_state     = optimizer_b.init(params)
    train_step_B  = make_train_step_unfreeze(pad_idx, optimizer_b)
    best_val = float('inf'); best_params = params; no_imp = 0
    rng_key = jax.random.PRNGKey(1234)

    for epoch in range(1, EPOCHS_B+1):
        t0 = time.time()
        perm = np.random.permutation(len(train_arr))
        tot = 0.0; nb=0
        for i in range(0, len(train_arr), bs):
            idx = perm[i:i+bs]
            if len(idx) == 0: continue
            batch = jnp.asarray(train_arr[idx])
            rng_key, subk = jax.random.split(rng_key)
            params, opt_state, loss = train_step_B(params, opt_state, batch, subk, DROPOUT_B, SMOOTH_B)
            tot += float(loss); nb += 1
        train_ppl = float(np.exp(tot/nb)) if nb>0 else float('inf')
        val_ppl = compute_perplexity(params, val_arr, pad_idx, batch=min(bs, len(val_arr)))
        dt = time.time()-t0
        print(f"[B] {epoch:03d}/{EPOCHS_B} | train {train_ppl:.4f} | val {val_ppl:.4f} | {dt:.1f}s")
        if val_ppl + 1e-3 < best_val:
            best_val = val_ppl; best_params = params; no_imp = 0
        else:
            no_imp += 1
            if no_imp >= PATIENCE_B:
                print("Early stop Phase B.")
                break
    params = best_params

    # save
    with open(TUNED_MODEL, "wb") as f:
        pickle.dump((params, processor, model_len), f, protocol=pickle.HIGHEST_PROTOCOL)
    size_mb = os.path.getsize(TUNED_MODEL)/1024/1024
    print(f"✅ Fine-tune done. Saved best → {TUNED_MODEL} ({size_mb:.2f} MB)")

if __name__ == "__main__":
    main()


# Multiple finetuning peptides new needs test

In [ ]:
# -*- coding: utf-8 -*-
# ==============================================================================
#   Multi-Specialist Fine-Tuning (from MASTER tuned)
#   - Loads a single tuned base (compact MERA-MPS)
#   - Per window [Lmin,Lmax]:
#       * filter Pfam within window + upsample near anchor (±Δ)
#       * set specialist MAX_LEN at anchor (even)
#       * Phase A: freeze encoder + (optionally) head
#       * Phase B: unfreeze encoder; head usually frozen for calibration
#   - Saves one specialist .pkl per window + a JSON manifest (incl. best val PPL)
#   - Memory-safe (JAX preallocate OFF)
# ==============================================================================

import os, re, time, math, json, random, pickle, warnings
from typing import List, Tuple, Optional

# ---- JAX memory safety (set BEFORE importing jax) ----
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.80")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "default")

import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
import jax.tree_util as jtu
import optax
warnings.filterwarnings("ignore", category=UserWarning)

# ========================== USER SETTINGS ====================================
BASE_MODEL   = os.environ.get("MASTER_TUNED", "/kaggle/input/alltrainedmodels/other/default/1/production_model_tuned.pkl")
OUT_DIR      = os.environ.get("SPECIALISTS_DIR", "/kaggle/working/specialists")
os.makedirs(OUT_DIR, exist_ok=True)

PFAM_FILES   = [
    "/kaggle/input/pfaseed/Pfam-A.seed",
    "/kaggle/input/pfafull/Pfam-A.full",
]

# --- Specialist windows (καλύπτουν καλά AMPs/short peptides)
SPECIALIST_WINDOWS: List[Tuple[int,int]] = [
    (24, 36),   # anchor ≈ 30/32
    (30, 48),   # anchor ≈ 40
    (44, 60),   # anchor ≈ 52/54
]

# Anchor selection for specialist MAX_LEN (must be even): "center_even" | "max_even"
ANCHOR_POLICY = "center_even"

# Dataset shaping inside each window
NEAR_DELTA             = 2       # upsample γύρω από anchor±Δ
NEAR_TARGET_MIN        = 1600    # ελάχιστα δείγματα κοντά στο anchor (με repeats)
HARD_CAP_MAX_LEN       = 256     # parser cap (SOS/EOS safety)
MIN_SAMPLES_REQUIRED   = 800     # αν < skip το window (για σταθερό training)

# Training global
TOKENS_PER_BATCH  = 8192
BATCH_MAX         = 64
PY_SEED           = 0

# Phase A (freeze encoder + head)
EPOCHS_A   = 10
LR_A       = 1.0e-4
DROPOUT_A  = 0.10
SMOOTH_A   = 0.02
PATIENCE_A = 4
FREEZE_HEAD_A = True   # shared head → σταθερή κλίμακα

# Phase B (unfreeze encoder); head συμπεριφορά
EPOCHS_B   = 100
LR_B       = 8.0e-5
DROPOUT_B  = 0.10
SMOOTH_B   = 0.02
PATIENCE_B = 8
FREEZE_HEAD_B = True   # προτείνεται TRUE για calibration consistency
HEAD_L2_B  = 0.0       # αν ξεκλειδώσεις head, βάλε π.χ. 1e-4

# ========================= SANITY / HELPERS ==================================
def is_compact(params):
    try:
        return ('encoder' in params and 'decoder' in params and
                'projection_matrix' in params and 'output_projection' in params and
                isinstance(params['encoder'], dict) and isinstance(params['decoder'], dict))
    except Exception:
        return False

def even(x: int) -> int:
    return x if x % 2 == 0 else (x+1)

def anchor_for_window(lo: int, hi: int, policy: str="center_even") -> int:
    if policy == "max_even":
        return even(hi)
    mid = (lo + hi) // 2
    return even(mid)

# ========================= DATA HELPERS ======================================
VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

def _is_seq_line(line: str) -> bool:
    if (not line) or line.startswith('#') or line.startswith('//'):
        return False
    parts = line.split()
    if len(parts) < 2:
        return False
    seq = parts[-1]
    return bool(re.fullmatch(r'[A-Za-z\-\.]+', seq))

def read_pfam_sequences(file_paths: List[str], max_len_cap: int=256) -> List[List[str]]:
    seqs = []
    for path in file_paths:
        if not path or not os.path.exists(path):
            print(f"⚠️  Missing file: {path}")
            continue
        try:
            with open(path, 'r', encoding='utf-8') as f:
                for raw in f:
                    line = raw.strip()
                    if not _is_seq_line(line):
                        continue
                    s = line.split()[-1].upper()
                    s = ''.join([c for c in s if c in VALID_AA])
                    if len(s) >= 1:
                        s = s[:max_len_cap-2]   # -2 for SOS/EOS
                        seqs.append(list(s))
        except Exception as e:
            print(f"⚠️  Failed to read {path}: {e}")
    # dedupe
    uniq, seen = [], set()
    for lst in seqs:
        ss = ''.join(lst)
        if ss not in seen:
            seen.add(ss); uniq.append(lst)
    return uniq

def filter_window(uniq: List[List[str]], lo: int, hi: int) -> List[List[str]]:
    return [s for s in uniq if lo <= len(s) <= hi]

def upsample_near_anchor(seqs: List[List[str]], anchor: int, delta: int, min_near: int, seed: int=123) -> List[List[str]]:
    rng = random.Random(seed)
    near = [s for s in seqs if (anchor - delta) <= len(s) <= (anchor + delta)]
    added = 0
    if len(near) > 0 and len(near) < min_near:
        need = min_near - len(near)
        seqs = list(seqs) + [rng.choice(near) for _ in range(need)]
        added = need
    print(f"  ↪ upsample near {anchor}±{delta}: +{added} (final {len(seqs)})")
    return seqs

# ========================= PACKING ===========================================
class DataProcessor:
    """Minimal shim for unpickling + vocab access, kept from base checkpoint."""
    def sequence_to_indices(self, seq):
        vocab = getattr(self, "vocab", {}) or {}
        unk = vocab.get("<UNK>", 0)
        return np.asarray([vocab.get(tok, unk) for tok in seq], dtype=np.int32)

def pad_and_pack(processor, seq_chars: List[str], max_len: int):
    sos=processor.vocab['<SOS>']; eos=processor.vocab['<EOS>']; pad=processor.vocab['<PAD>']
    seq = ['<SOS>'] + list(seq_chars)[:max_len-2] + ['<EOS>']
    idx = processor.sequence_to_indices(seq)
    if len(idx) < max_len:
        idx = np.concatenate([idx, np.full((max_len-len(idx),), pad, np.int32)])
    return idx

def build_arrays(processor, seq_lists: List[List[str]], max_len: int):
    if len(seq_lists) == 0:
        return np.empty((0, max_len), dtype=np.int32)
    return np.asarray([pad_and_pack(processor, s, max_len) for s in seq_lists], dtype=np.int32)

# ========================= MODEL SHAPE OPS ===================================
def set_pad_identity(params, pad_idx: int):
    mps_enc = params['encoder']['mps']
    mps_dec = params['decoder']['mps']
    L, V, chi, _ = mps_enc.shape
    I = jnp.eye(chi, dtype=mps_enc.dtype)
    params['encoder']['mps'] = mps_enc.at[:, pad_idx].set(I)
    params['decoder']['mps'] = mps_dec.at[:, pad_idx].set(I)

def grow_params(params, old_len, new_len, bond_dim_mps, bond_dim_mera, vocab_size, rng):
    assert new_len > old_len and new_len % 2 == 0
    def grow_block(block, rng_block):
        mps_old = block['mps']; isos_old = block['isos']
        add_L = new_len - old_len
        k1, k2 = jax.random.split(rng_block)
        mps_new = jnp.zeros((new_len, vocab_size, bond_dim_mps, bond_dim_mps), dtype=mps_old.dtype)
        mps_new = mps_new.at[:old_len].set(mps_old * 0.999)
        if add_L > 0:
            init_tail = jax.random.normal(k1, (add_L, vocab_size, bond_dim_mps, bond_dim_mps)) * 0.1
            mps_new = mps_new.at[old_len:].set(init_tail)
        old_d = old_len // 2; new_d = new_len // 2
        isos_new = jnp.zeros((new_d, bond_dim_mps, bond_dim_mps, bond_dim_mera), dtype=isos_old.dtype)
        isos_new = isos_new.at[:old_d].set(isos_old * 0.999)
        if new_d - old_d > 0:
            init_iso_tail = jax.random.normal(k2, (new_d - old_d, bond_dim_mps, bond_dim_mps, bond_dim_mera)) * 0.1
            isos_new = isos_new.at[old_d:].set(init_iso_tail)
        return {'mps': mps_new, 'isos': isos_new}
    rng_enc, rng_dec, kproj = jax.random.split(rng, 3)
    new_encoder = grow_block(params['encoder'], rng_enc)
    new_decoder = grow_block(params['decoder'], rng_dec)
    old_td = (old_len//2) * bond_dim_mera
    new_td = (new_len//2) * bond_dim_mera
    proj_old = params['projection_matrix']
    proj_new = jnp.zeros((new_td, bond_dim_mps), dtype=proj_old.dtype)
    k = min(old_td, new_td)
    proj_new = proj_new.at[:k, :].set(proj_old[:k, :])
    if new_td > k:
        init_tail = jax.random.normal(kproj, (new_td - k, bond_dim_mps)) * 0.1
        proj_new = proj_new.at[k:, :].set(init_tail)
    return {
        'encoder': new_encoder,
        'decoder': new_decoder,
        'projection_matrix': proj_new,
        'output_projection': params['output_projection'],
    }

def shrink_params(params, old_len, new_len):
    assert new_len < old_len and new_len % 2 == 0
    def shrink_block(block):
        mps_old = block['mps']; isos_old = block['isos']
        return {'mps': mps_old[:new_len], 'isos': isos_old[:(new_len//2)]}
    new_encoder = shrink_block(params['encoder'])
    new_decoder = shrink_block(params['decoder'])
    old_td = (old_len//2) * params['encoder']['isos'].shape[-1]
    new_td = (new_len//2) * new_encoder['isos'].shape[-1]
    proj_old = params['projection_matrix']
    proj_new = proj_old[:new_td]
    return {
        'encoder': new_encoder,
        'decoder': new_decoder,
        'projection_matrix': proj_new,
        'output_projection': params['output_projection'],
    }

# ========================= ENCODE/DECODE (compact) ===========================
def _encode_step(carry, token_and_pos):
    state, mps, pos, rng_key, rate = carry
    tok, p = token_and_pos
    W = mps[p, tok]
    state = state @ W
    key = jax.random.fold_in(rng_key, p)
    keep = 1.0 - rate
    mask = jax.random.bernoulli(key, p=jnp.clip(keep, 0.0, 1.0), shape=state.shape)
    state = jnp.where(mask, state / jnp.maximum(keep, 1e-6), 0.0)
    return (state, mps, p+1, rng_key, rate), state

@jax.jit
def encode_compact(enc_params, seq_indices, rng_key, dropout_rate):
    L = enc_params['mps'].shape[0]
    chi = enc_params['mps'].shape[-1]
    state0 = jnp.zeros((chi,), dtype=jnp.float32).at[0].set(1.0)
    positions = jnp.arange(L, dtype=jnp.int32)
    (_, _, _, _, _), states = lax.scan(
        _encode_step,
        (state0, enc_params['mps'], 0, rng_key, dropout_rate),
        (seq_indices, positions)
    )
    def pair_reduce(_, i):
        l = states[2*i]; r = states[2*i+1]
        iso = enc_params['isos'][i]
        hi  = jnp.einsum('i,j,ijk->k', l, r, iso)
        return None, hi
    _, hi_list = lax.scan(pair_reduce, None, jnp.arange(L//2))
    tv = hi_list.reshape(-1)
    return tv / (jnp.linalg.norm(tv) + 1e-9)

def _decode_step(carry, tok_and_pos):
    vec, ctx, mps_dec, out_proj, first_flag = carry
    tok, p = tok_and_pos
    W = mps_dec[p, tok]
    vec = vec @ W
    vec = jnp.where(first_flag, vec + ctx, vec)
    logits = vec @ out_proj
    return (vec, ctx, mps_dec, out_proj, False), logits

@jax.jit
def decode_tf_compact(params, thought_vector, decoder_input):
    L = params['decoder']['mps'].shape[0]
    chi = params['decoder']['mps'].shape[-1]
    ctx = thought_vector @ params['projection_matrix']
    vec0 = jnp.zeros((chi,), dtype=jnp.float32).at[0].set(1.0)
    positions = jnp.arange(L-1, dtype=jnp.int32)
    mps_dec = params['decoder']['mps'][:-1]
    (_, _, _, _, _), logits = lax.scan(
        _decode_step,
        (vec0, ctx, mps_dec, params['output_projection'], True),
        (decoder_input, positions)
    )
    return logits

# ========================= LOSS / TRAIN ======================================
def _xent_logits_with_int_labels(logits, labels_int, label_smoothing=0.0):
    V = logits.shape[-1]
    ls = jnp.asarray(label_smoothing, dtype=logits.dtype)
    onehot = jax.nn.one_hot(labels_int, V)
    smoothed = optax.smooth_labels(onehot, ls)
    return optax.softmax_cross_entropy(logits=logits, labels=smoothed)

@jax.jit
def per_seq_losses(params, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing):
    B = batch_inputs.shape[0]
    enc_keys = jax.random.split(rng_key, B)
    tvs = jax.vmap(encode_compact, in_axes=(None,0,0,None))(
        params['encoder'], batch_inputs, enc_keys, dropout_rate
    )
    dec_in  = batch_inputs[:, :-1]
    dec_tgt = batch_inputs[:, 1:]
    logits  = jax.vmap(decode_tf_compact, in_axes=(None,0,0))(params, tvs, dec_in)
    token_losses = _xent_logits_with_int_labels(logits, dec_tgt, label_smoothing)
    mask = (dec_tgt != pad_idx)
    per_seq_sum = (token_losses * mask).sum(axis=1)
    per_seq_cnt = mask.sum(axis=1)
    return per_seq_sum / jnp.maximum(per_seq_cnt, 1)

@jax.jit
def loss_fn(params, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing):
    losses = per_seq_losses(params, batch_inputs, pad_idx, rng_key, dropout_rate, label_smoothing)
    counts = (batch_inputs[:, 1:] != pad_idx).sum(axis=1).astype(jnp.float32)
    denom = jnp.maximum(counts.sum(), 1.0)
    return (losses * counts).sum() / denom

def compute_batch_size(max_len: int) -> int:
    return max(8, min(BATCH_MAX, max(1, TOKENS_PER_BATCH // max(1, max_len))))

def make_optimizer(lr: float, head_l2: float = 0.0):
    opt = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=lr, weight_decay=2e-4)
    )
    return opt

def compute_perplexity(params, data_arr, pad_idx, batch=64):
    n = len(data_arr)
    if n == 0:
        return float('inf')
    batch = max(1, min(batch, n))
    rng = jax.random.PRNGKey(999)
    tot_ll = 0.0; tot_tokens = 0
    for i in range(0, n, batch):
        chunk_np = np.asarray(data_arr[i:i+batch])
        chunk = jnp.asarray(chunk_np)
        l = float(loss_fn(params, chunk, pad_idx, rng, 0.0, 0.0))
        nonpad = int((chunk_np[:, 1:] != pad_idx).sum())
        tot_ll += l * nonpad
        tot_tokens += nonpad
    if tot_tokens == 0:
        return float('inf')
    return float(np.exp(tot_ll / tot_tokens))

# ---------- train steps ----------
def make_train_step_freeze(pad_idx, optimizer, freeze_encoder=True, freeze_head=True, head_l2=0.0):
    @jax.jit
    def step(params, opt_state, batch, rng_key, dropout_rate, ls):
        l, grads = jax.value_and_grad(loss_fn)(
            params, batch, pad_idx, rng_key, dropout_rate, ls
        )
        if head_l2 > 0.0 and not freeze_head:
            l2 = 0.5 * head_l2 * jnp.sum(params['output_projection']**2)
            l = l + l2
            g_head = grads['output_projection'] + head_l2 * params['output_projection']
            grads = dict(grads); grads['output_projection'] = g_head
        if freeze_encoder:
            grads = dict(grads); grads['encoder'] = jtu.tree_map(jnp.zeros_like, grads['encoder'])
        if freeze_head:
            grads = dict(grads); grads['output_projection'] = jnp.zeros_like(grads['output_projection'])
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, l
    return step

def make_train_step_unfreeze(pad_idx, optimizer, freeze_head=True, head_l2=0.0):
    @jax.jit
    def step(params, opt_state, batch, rng_key, dropout_rate, ls):
        l, grads = jax.value_and_grad(loss_fn)(
            params, batch, pad_idx, rng_key, dropout_rate, ls
        )
        if head_l2 > 0.0 and not freeze_head:
            l2 = 0.5 * head_l2 * jnp.sum(params['output_projection']**2)
            l = l + l2
            grads = dict(grads)
            grads['output_projection'] = grads['output_projection'] + head_l2 * params['output_projection']
        if freeze_head:
            grads = dict(grads)
            grads['output_projection'] = jnp.zeros_like(grads['output_projection'])
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, l
    return step

# ========================= MULTI-FINETUNE LOOP ===============================
def main():
    def now(): return time.strftime("%Y-%m-%d %H:%M:%S")
    random.seed(PY_SEED); np.random.seed(PY_SEED)

    if not os.path.exists(BASE_MODEL):
        raise FileNotFoundError(f"Base model not found: {BASE_MODEL}")
    with open(BASE_MODEL, "rb") as f:
        payload = pickle.load(f)
    if isinstance(payload, tuple) and len(payload) >= 3:
        base_params, processor, base_len = payload[:3]
    else:
        raise RuntimeError("Unsupported base model pickle format.")
    if not is_compact(base_params):
        raise RuntimeError("Only compact param format supported.")

    chi_mps = int(base_params['encoder']['mps'].shape[-1])
    chim    = int(base_params['encoder']['isos'].shape[-1])
    vocab   = int(base_params['encoder']['mps'].shape[1])
    print(f"▶ Loaded MASTER ✓  len={base_len}  χ={chi_mps}  chim={chim}  vocab={vocab}")

    all_seqs = read_pfam_sequences(PFAM_FILES, max_len_cap=HARD_CAP_MAX_LEN)
    print(f"Candidates (deduped): {len(all_seqs)}")

    manifest = {"base_model": BASE_MODEL, "specialists": []}

    for (Lmin, Lmax) in SPECIALIST_WINDOWS:
        Lmin_i, Lmax_i = int(Lmin), int(Lmax)
        L_anchor = anchor_for_window(Lmin_i, Lmax_i, ANCHOR_POLICY)
        if L_anchor % 2 != 0: L_anchor += 1
        if L_anchor < 2: L_anchor = 2

        # track best val PPL per phase for manifest
        best_val_A = None
        best_val_B = None

        print("\n" + "="*74)
        print(f"  Specialist window [{Lmin_i}, {Lmax_i}]  | anchor L={L_anchor} (policy: {ANCHOR_POLICY})")
        print("="*74)

        seqs = filter_window(all_seqs, Lmin_i, Lmax_i)
        print(f"  raw in-window: {len(seqs)}")
        seqs = upsample_near_anchor(seqs, L_anchor, NEAR_DELTA, NEAR_TARGET_MIN, seed=123)
        if len(seqs) < MIN_SAMPLES_REQUIRED:
            print(f"  ⚠️  Too few samples ({len(seqs)}) — skipping this window.")
            continue

        rng = random.Random(1337 + L_anchor)
        rng.shuffle(seqs)
        n_val = max(200, int(0.2 * len(seqs)))
        train_seqs = seqs[:-n_val]
        val_seqs   = seqs[-n_val:]

        params = pickle.loads(pickle.dumps(base_params, protocol=pickle.HIGHEST_PROTOCOL))
        model_len = int(params['encoder']['mps'].shape[0])

        if model_len != L_anchor:
            if model_len > L_anchor:
                print(f"  Shrinking model parameters: {model_len} → {L_anchor}")
                params = shrink_params(params, model_len, L_anchor)
            else:
                print(f"  Growing model parameters: {model_len} → {L_anchor}")
                params = grow_params(params, model_len, L_anchor, chi_mps, chim, vocab, jax.random.PRNGKey(99))
            model_len = L_anchor

        pad_idx = processor.vocab['<PAD>']
        set_pad_identity(params, pad_idx)

        train_arr = build_arrays(processor, train_seqs, model_len)
        val_arr   = build_arrays(processor, val_seqs,   model_len)
        bs = compute_batch_size(model_len)
        print(f"  window unique={len(seqs)} | train={len(train_arr)} val={len(val_arr)} | batch≈{bs}")

        # Warmup JIT
        if len(train_arr) >= 1:
            dummy = jnp.asarray(train_arr[:max(1, min(8, len(train_arr)))])
            _ = loss_fn(params, dummy, pad_idx, jax.random.PRNGKey(7), 0.0, 0.0).block_until_ready()

        # ---------- Phase A ----------
        if EPOCHS_A > 0:
            optimizer_a  = make_optimizer(LR_A)
            opt_state    = optimizer_a.init(params)
            step_A       = make_train_step_freeze(
                pad_idx, optimizer_a,
                freeze_encoder=True,
                freeze_head=FREEZE_HEAD_A,
                head_l2=0.0
            )
            best_val = float('inf'); best_params = params; no_imp = 0
            rng_key = jax.random.PRNGKey(0)

            for epoch in range(1, EPOCHS_A+1):
                t0 = time.time()
                perm = np.random.permutation(len(train_arr))
                tot = 0.0; nb=0; total_tokens=0
                for i in range(0, len(train_arr), bs):
                    idx = perm[i:i+bs]
                    if len(idx) == 0: continue
                    batch = jnp.asarray(train_arr[idx])
                    rng_key, subk = jax.random.split(rng_key)
                    params, opt_state, loss = step_A(params, opt_state, batch, subk, DROPOUT_A, SMOOTH_A)
                    tot += float(loss); nb += 1
                    total_tokens += int((np.asarray(train_arr[idx])[:,1:] != pad_idx).sum())
                train_ppl = float(np.exp(tot/nb)) if nb>0 else float('inf')
                val_ppl = compute_perplexity(params, val_arr, pad_idx, batch=min(bs, len(val_arr)))
                dt = time.time()-t0
                print(f"  [A] {epoch:03d}/{EPOCHS_A} | train {train_ppl:.4f} | val {val_ppl:.4f} | {dt:.1f}s")
                if val_ppl + 1e-3 < best_val:
                    best_val = val_ppl; best_params = params; no_imp = 0
                else:
                    no_imp += 1
                    if no_imp >= PATIENCE_A:
                        print("  Early stop Phase A.")
                        break
            best_val_A = float(best_val)
            params = best_params

        # ---------- Phase B ----------
        if EPOCHS_B > 0:
            optimizer_b  = make_optimizer(LR_B, head_l2=HEAD_L2_B)
            opt_state    = optimizer_b.init(params)
            step_B       = make_train_step_unfreeze(
                pad_idx, optimizer_b,
                freeze_head=FREEZE_HEAD_B,
                head_l2=HEAD_L2_B
            )
            best_val = float('inf'); best_params = params; no_imp = 0
            rng_key = jax.random.PRNGKey(1234)

            for epoch in range(1, EPOCHS_B+1):
                t0 = time.time()
                perm = np.random.permutation(len(train_arr))
                tot = 0.0; nb=0
                for i in range(0, len(train_arr), bs):
                    idx = perm[i:i+bs]
                    if len(idx) == 0: continue
                    batch = jnp.asarray(train_arr[idx])
                    rng_key, subk = jax.random.split(rng_key)
                    params, opt_state, loss = step_B(params, opt_state, batch, subk, DROPOUT_B, SMOOTH_B)
                    tot += float(loss); nb += 1
                train_ppl = float(np.exp(tot/nb)) if nb>0 else float('inf')
                val_ppl = compute_perplexity(params, val_arr, pad_idx, batch=min(bs, len(val_arr)))
                dt = time.time()-t0
                print(f"  [B] {epoch:03d}/{EPOCHS_B} | train {train_ppl:.4f} | val {val_ppl:.4f} | {dt:.1f}s")
                if val_ppl + 1e-3 < best_val:
                    best_val = val_ppl; best_params = params; no_imp = 0
                else:
                    no_imp += 1
                    if no_imp >= PATIENCE_B:
                        print("  Early stop Phase B.")
                        break
            best_val_B = float(best_val)
            params = best_params

        # ---------- Save specialist ----------
        tag = f"win{int(Lmin)}-{int(Lmax)}_len{int(L_anchor)}"
        out_path = os.path.join(OUT_DIR, f"specialist_{tag}.pkl")
        with open(out_path, "wb") as f:
            pickle.dump((params, processor, int(L_anchor)), f, protocol=pickle.HIGHEST_PROTOCOL)
        size_mb = os.path.getsize(out_path)/1024/1024
        print(f"  ✅ Saved specialist → {out_path} ({size_mb:.2f} MB)")

        # Επιλογή καλύτερου validation perplexity για το manifest
        val_ppl_best = best_val_B if (best_val_B is not None) else best_val_A
        if val_ppl_best is None:
            val_ppl_best = float("nan")

        manifest["specialists"].append({
            "window": [int(Lmin), int(Lmax)],
            "anchor_len": int(L_anchor),
            "path": out_path,
            "val_ppl_best": float(val_ppl_best),
            "freeze_head_A": bool(FREEZE_HEAD_A),
            "freeze_head_B": bool(FREEZE_HEAD_B),
            "head_L2_B": float(HEAD_L2_B),
            "epochs_A": int(EPOCHS_A),
            "epochs_B": int(EPOCHS_B),
            "near_delta": int(NEAR_DELTA),
            "near_target_min": int(NEAR_TARGET_MIN),
            "batch_tokens": int(TOKENS_PER_BATCH),
        })

    # Save manifest
    manifest_path = os.path.join(OUT_DIR, "specialists_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f"\n📄 Manifest saved: {manifest_path}")
    print("Done.")

if __name__ == "__main__":
    main()


In [2]:
!pip freeze > kaggle_requirements.txt


# # pep_product 

In [1]:
%%writefile /kaggle/working/pep_product.py
# -*- coding: utf-8 -*-
"""
pep_product.py — Robust peptide AMP + MIC-ready (classification + regression) + Δ-MutScan

What you get
------------
• Specialist routing + projection bagging (n=5) για σταθερά embeddings
• GroupKFold by 3-mer signature (leakage-safe folds)
• Nested calibration + precision-targeted threshold (precision≥0.80 by default)
• Split-conformal margin για principled abstention
• Inference-time Δ-MutScan (fast/full) για single-aa robustness
• Νέα features: helical hydrophobic moment (μH), amphipathic index, Cys-topology/motifs
• Guardrails (CPP/hydrophobic traps, length), deterministic free tier
• CSV/FASTA batch prediction, καθαρό CLI, JSON training summaries
• Classification heads (π.χ. AMP/Toxicity/Stability/CPP) + Regression heads (π.χ. MIC)
"""

import os, re, json, time, glob, pickle, hashlib, warnings, argparse, math, joblib
from typing import List, Dict, Any, Tuple

# ---- JAX memory safety (set BEFORE importing jax) ----
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.80")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "default")

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
from jax import lax

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, HuberRegressor
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, brier_score_loss,
    precision_recall_curve, mean_absolute_error, r2_score
)

warnings.filterwarnings("ignore", category=UserWarning)


class DataProcessor:
    """Minimal shim: αρκεί για unpickle + vocab + packing."""
    def sequence_to_indices(self, seq):
        vocab = getattr(self, "vocab", {}) or {}
        unk = vocab.get("<UNK>", 0)
        return np.asarray([vocab.get(tok, unk) for tok in seq], dtype=np.int32)


# ==== Adapter helpers (Platt-style) ====
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def _safe_logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def apply_adapter_to_df(df: pd.DataFrame, adapter_cfg: dict, head_name: str):
    """
    Προσάρμοσε p με p_cal = sigmoid(a * logit(p) + b).
    - Αν adapter_cfg έχει 'head', εφαρμόζεται ΜΟΝΟ σ' αυτό το head.
    - Αλλιώς εφαρμόζεται σε ΟΛΕΣ τις στήλες που τελειώνουν σε '_prob'.
    - Αν υπάρχει 'thr' στο adapter, φτιάχνει *_prob_cal, *_label_cal, *_decision_cal.
    """
    a = float(adapter_cfg["a"]); b = float(adapter_cfg["b"])
    target_head = adapter_cfg.get("head", None)
    thr = adapter_cfg.get("thr", None)

    def _one(colprob, base_name):
        p = df[colprob].astype(float).to_numpy()
        p_cal = _sigmoid(a * _safe_logit(p) + b)
        df[f"{base_name}_prob_cal"] = p_cal
        if thr is not None:
            df[f"{base_name}_label_cal"] = (p_cal >= float(thr)).astype(int)
            if f"{base_name}_decision" in df.columns:
                dec = df[f"{base_name}_decision"].astype(str).to_numpy()
                dec = np.where(dec=="abstain", "abstain",
                               np.where(p_cal >= float(thr), "positive", "negative"))
                df[f"{base_name}_decision_cal"] = dec
        return df

    if target_head:
        base = target_head
        colprob = f"{base}_prob"
        if colprob in df.columns:
            df = _one(colprob, base)
    else:
        for col in list(df.columns):
            if col.endswith("_prob"):
                base = col[:-5]
                df = _one(col, base)
    return df


# ===================== DEFAULT CONFIG =====================
SPECIALISTS_MANIFEST = os.environ.get(
    "SPECIALISTS_MANIFEST", "/kaggle/input/latestspecialists/pytorch/default/1/specialists_manifest.json"
)
SPECIALIST_PKLS_GLOB = os.environ.get(
    "SPECIALIST_PKLS_GLOB", "/kaggle/input/latestspecialists/pytorch/default/1/specialist_*.pkl"
)

HEADS_DIR = os.environ.get("HEADS_DIR", "/kaggle/working/heads")
os.makedirs(HEADS_DIR, exist_ok=True)

PROJ_DIM = int(os.environ.get("PROJ_DIM", "128"))
BLEND_POLICY = os.environ.get("BLEND_POLICY", "triangular")
RNG_SEED = 42
CV_FOLDS = int(os.environ.get("CV_FOLDS", "5"))
PRECISION_TARGET = float(os.environ.get("PRECISION_TARGET", "0.80"))
CONFORMAL_ALPHA = float(os.environ.get("CONFORMAL_ALPHA", "0.02"))
PROJ_BAG = int(os.environ.get("PROJ_BAG", "5"))
MIN_TRAIN_SAMPLES = int(os.environ.get("MIN_TRAIN_SAMPLES", "30"))

# ===================== AA maps & heuristics =====================
AA20 = "ACDEFGHIKLMNPQRSTVWY"
AA2IDX = {a: i for i, a in enumerate(AA20)}

# Kyte-Doolittle hydrophobicities
KD = {
    'I':4.5,'V':4.2,'L':3.8,'F':2.8,'C':2.5,'M':1.9,'A':1.8,'G':-0.4,'T':-0.7,'S':-0.8,
    'W':-0.9,'Y':-1.3,'P':-1.6,'H':-3.2,'E':-3.5,'Q':-3.5,'D':-3.5,'N':-3.5,'K':-3.9,'R':-4.5
}
HYDRO = set("AFILMVWYGC")

# For Δ-MutScan (fast mode) — biochemical neighbors
AA_NEIGHBORS = {
    "D":["E","N"], "E":["D","Q"], "K":["R","H"], "R":["K","H"], "H":["K","R"],
    "N":["D","Q","S"], "Q":["E","N","T"], "S":["T","A","N"], "T":["S","A","Q"],
    "A":["S","T","G","V"], "G":["A","S"], "V":["L","I","A"], "L":["I","V","M"],
    "I":["L","V","M"], "M":["L","I"], "F":["Y","W","L"], "Y":["F","W","H"],
    "W":["F","Y"], "C":["A","S"], "P":["A","S"]
}

# ===================== Basic feature helpers =====================
def aa_composition(seq: str) -> np.ndarray:
    v = np.zeros((20,), np.float32); L = max(1, len(seq))
    for ch in seq:
        i = AA2IDX.get(ch, None)
        if i is not None: v[i] += 1.0
    return v / float(L)

def gravy(seq: str) -> float:
    if not seq: return 0.0
    return float(np.mean([KD.get(ch, 0.0) for ch in seq]))

def net_charge_pH7(seq: str) -> float:
    pos = sum(ch in "KR" for ch in seq) + 0.1 * sum(ch == 'H' for ch in seq)
    neg = sum(ch in "DE" for ch in seq)
    return float(pos - neg)

def aromatic_fraction(seq: str) -> float:
    if not seq: return 0.0
    return sum(ch in "FWY" for ch in seq) / len(seq)

def hydro_fraction(seq: str) -> float:
    if not seq: return 0.0
    return sum(ch in HYDRO for ch in seq) / len(seq)

def max_hydrophobic_run(seq: str) -> int:
    m = 0; c = 0
    for ch in seq:
        if ch in HYDRO: c += 1; m = max(m, c)
        else: c = 0
    return m

# NEW: Helical hydrophobic moment (μH) and amphipathic index
_HELIX_RAD = math.radians(100.0)
def helical_moment(seq: str) -> float:
    x = y = 0.0
    for i, ch in enumerate(seq):
        k = KD.get(ch, 0.0)
        angle = i * _HELIX_RAD
        x += k * math.cos(angle); y += k * math.sin(angle)
    L = max(len(seq), 1)
    return float(math.sqrt(x*x + y*y) / L)

def amphipathic_index(seq: str) -> float:
    if not seq: return 0.0
    vals = []
    for i, ch in enumerate(seq):
        k = KD.get(ch, 0.0)
        ang = (i * _HELIX_RAD) % (2*math.pi)
        side = 1 if (ang < math.pi) else -1
        vals.append((side, k))
    s1 = [k for side, k in vals if side==1]
    s2 = [k for side, k in vals if side==-1]
    if not s1 or not s2:
        return 0.0
    return float(abs(np.mean(s1) - np.mean(s2)))

# NEW: motifs & Cys topology
MOTIFS = ["W..W", "KWK", "KLAK", "LRLR", "GLGF", "G...G"]
def motif_hits(seq: str) -> int:
    cnt = 0
    for m in MOTIFS:
        pattern = m.replace(".", "[A-Z]")
        cnt += len(re.findall(pattern, seq))
    return cnt

def cysteine_features(seq: str) -> Tuple[int,int,float]:
    idx = [i for i,ch in enumerate(seq) if ch=="C"]
    nC = len(idx)
    pairs = nC//2
    mean_gap = float(np.mean(np.diff(idx))) if len(idx)>=2 else 0.0
    return nC, pairs, mean_gap

def amp_activity_proxy_from_heuristics(seq: str) -> float:
    L = len(seq)
    charge = max(0.0, min(12.0, net_charge_pH7(seq)))
    hydro = hydro_fraction(seq)
    arom = aromatic_fraction(seq)
    length_ok = 1.0 if 10 <= L <= 40 else max(0.0, 1.0 - abs(L - 25.0) / 25.0)
    raw = 0.50 * (charge / 12.0) + 0.35 * hydro + 0.05 * (1.0 - arom) + 0.10 * helical_moment(seq)
    return float(np.clip(raw * length_ok, 0.0, 1.0))

def admet_scores_from_heuristics(seq: str) -> Tuple[float, float, float]:
    g = gravy(seq); h = hydro_fraction(seq); a = aromatic_fraction(seq)
    r = float(max_hydrophobic_run(seq)); q = abs(net_charge_pH7(seq))
    sol = (-g) + (-0.2 * r) + (1.0 - h)
    stab = 0.4 * h + 0.1 * a - 0.2 * q
    agg = 0.6 * h + 0.3 * r + 0.1 * a
    return (float(sol), float(stab), float(agg))

# ===================== Specialists I/O =====================
class ProcShim:
    def sequence_to_indices(self, seq):
        vocab = getattr(self, "vocab", {}) or {}
        unk = vocab.get("<UNK>", 0)
        return np.asarray([vocab.get(tok, unk) for tok in seq], dtype=np.int32)

def load_manifest(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        man = json.load(f)
    return list(man.get("specialists", []))

def load_specialist_pkl(pkl_path: str):
    with open(pkl_path, "rb") as f:
        payload = pickle.load(f)
    if isinstance(payload, tuple) and len(payload) >= 3:
        params, processor, max_len = payload[:3]
    else:
        raise RuntimeError(f"Unsupported pickle format: {pkl_path}")
    if not hasattr(processor, 'vocab'):
        processor = ProcShim()
    chi_mps = int(params['encoder']['mps'].shape[-1])
    return {
        "path": pkl_path, "params": params, "processor": processor,
        "max_len": int(max_len), "anchor_len": int(max_len), "window": (1, max_len - 2),
        "bond_dim_mps": chi_mps,
    }

def load_all_specialists() -> List[Dict[str, Any]]:
    specs = []
    if os.path.exists(SPECIALISTS_MANIFEST):
        try:
            meta = load_manifest(SPECIALISTS_MANIFEST)
            for it in meta:
                path = it.get("path", "")
                if os.path.exists(path):
                    s = load_specialist_pkl(path)
                    s.update(it)
                    specs.append(s)
        except Exception as e:
            print(f"⚠️ manifest read failed: {e}")
    if not specs:
        for p in sorted(glob.glob(SPECIALIST_PKLS_GLOB)):
            try:
                specs.append(load_specialist_pkl(p))
            except Exception as e:
                print(f"⚠️ skip {p}: {e}")
    if not specs:
        raise RuntimeError("No specialists found.")
    return specs

# ===================== Encode/Decode primitives (COMPACT) =====================
def pad_and_pack(processor, seq_chars, max_len):
    sos=processor.vocab['<SOS>']; eos=processor.vocab['<EOS>']; pad=processor.vocab['<PAD>']
    seq = ['<SOS>'] + list(seq_chars)[:max_len-2] + ['<EOS>']
    idx = processor.sequence_to_indices(seq)
    if len(idx) < max_len:
        idx = np.concatenate([idx, np.full((max_len-len(idx),), pad, np.int32)])
    return idx

@jax.jit
def encode_compact(enc_params, seq_indices):
    L, chi = enc_params['mps'].shape[0], enc_params['mps'].shape[-1]
    state0 = jnp.zeros((chi,), dtype=jnp.float32).at[0].set(1.0)
    def scan_body(carry, x):
        state, mps = carry
        tok, pos = x
        return (state @ mps[pos, tok], mps), state @ mps[pos, tok]
    (_, _), states = lax.scan(scan_body, (state0, enc_params['mps']), (seq_indices, jnp.arange(L)))
    def pair_reduce(_, i):
        return None, jnp.einsum('i,j,ijk->k', states[2*i], states[2*i+1], enc_params['isos'][i])
    _, hi_list = lax.scan(pair_reduce, None, jnp.arange(L//2))
    tv = hi_list.reshape(-1)
    return tv / (jnp.linalg.norm(tv) + 1e-9)

@jax.jit
def decode_teacher_forcing_compact(params, thought_vector, decoder_input):
    L, chi = params['decoder']['mps'].shape[0], params['decoder']['mps'].shape[-1]
    ctx = thought_vector @ params['projection_matrix']
    vec0 = jnp.zeros((chi,), dtype=jnp.float32).at[0].set(1.0)
    def scan_body(carry, x):
        vec, first_flag = carry
        tok, pos = x
        vec = vec @ params['decoder']['mps'][pos, tok]
        vec = jnp.where(first_flag, vec + ctx, vec)
        return (vec, False), vec @ params['output_projection']
    (_, _), logits = lax.scan(scan_body, (vec0, True), (decoder_input, jnp.arange(L-1)))
    return logits

# ===================== Routing & Projection bagging =====================
def pick_applicable_specialists(specs: List[Dict[str, Any]], L: int) -> List[Dict[str, Any]]:
    out = [s for s in specs if s.get("window",(1,1e9))[0] <= L <= s.get("window",(0,0))[1] and s["max_len"] >= (L + 2)]
    return out if out else [min(specs, key=lambda s: abs(s.get("anchor_len", s["max_len"]) - L))]

def blend_weights(L: int, specs: List[Dict[str, Any]], policy="triangular") -> np.ndarray:
    if len(specs) == 1 or policy == "hard":
        w = np.zeros(len(specs)); w[np.argmin([abs(s.get("anchor_len", s["max_len"]) - L) for s in specs])] = 1.0; return w
    if policy == "inverse_dist":
        dist = np.array([abs(s.get("anchor_len", s["max_len"]) - L) for s in specs], dtype=np.float64); w = 1.0 / (dist + 1e-6); return w / w.sum()
    ws = [max(0.0, 1.0 - abs(L - float(s.get("anchor_len", s["max_len"]))) / (max(2.0, (s.get("window",(0,0))[1] - s.get("window",(0,0))[0])) / 2.0 + 1e-6)) for s in specs]
    w = np.array(ws, np.float64); return w / w.sum() if w.sum() > 0 else np.ones_like(w) / len(w)

_proj_cache: Dict[str, np.ndarray] = {}
def _projs_for_specialist(spec_path: str, in_dim: int, out_dim: int, n: int) -> List[np.ndarray]:
    Ps = []
    for k in range(n):
        key = f"{spec_path}|{in_dim}|{out_dim}|{k}"
        if key in _proj_cache: Ps.append(_proj_cache[key]); continue
        seed = int.from_bytes(hashlib.sha256(key.encode()).digest()[:8], "little")
        rng = np.random.default_rng(seed)
        P = rng.normal(0.0, 1.0/np.sqrt(max(1,in_dim)), size=(in_dim,out_dim)).astype(np.float32)
        _proj_cache[key] = P; Ps.append(P)
    return Ps

# ===================== Feature extraction =====================
def features_for_sequence(seq: str, specialists: List[Dict[str, Any]], proj_dim: int, blend: str, proj_bag: int) -> Dict[str, Any]:
    L = len(seq)
    specs = pick_applicable_specialists(specialists, L)
    w = blend_weights(L, specs, blend) if specs else []

    scalars = {"mean_nll": [], "perplexity": [], "entropy": [], "margin": []}
    embeds = []
    for wi, s in zip(w, specs):
        arr = pad_and_pack(s["processor"], list(seq), s["max_len"])
        inp = jnp.asarray(arr)
        tv = encode_compact(s['params']['encoder'], inp)
        logits = decode_teacher_forcing_compact(s['params'], tv, inp[:-1])

        tgt, mask = inp[1:], (inp[1:] != s["processor"].vocab['<PAD>'])
        logp = jax.nn.log_softmax(logits, axis=-1)
        mean_nll = -float((logp[jnp.arange(len(tgt)), tgt] * mask).sum() / mask.sum())

        p = jnp.exp(logp); ent = -jnp.sum(p * logp, axis=-1)
        top2 = jax.lax.top_k(logits, 2)[0]

        scalars["mean_nll"].append(wi * mean_nll)
        scalars["perplexity"].append(wi * np.exp(mean_nll))
        scalars["entropy"].append(wi * float(ent[:L].mean()))
        scalars["margin"].append(wi * float((top2[:L,0]-top2[:L,1]).mean()))

        tv_np = np.array(tv, dtype=np.float32)
        for P in _projs_for_specialist(s["path"], tv_np.shape[0], proj_dim, n=proj_bag):
            emb = tv_np @ P; emb /= max(1e-9, float(np.linalg.norm(emb)))
            embeds.append(wi * emb)

    embed_vec = np.mean(np.stack(embeds, axis=0), axis=0).astype(np.float32) if embeds else np.zeros((proj_dim,), np.float32)

    sol_sc, stab_sc, agg_sc = admet_scores_from_heuristics(seq)
    muH = helical_moment(seq)
    amph = amphipathic_index(seq)
    nC, nC_pairs, c_gap = cysteine_features(seq)
    motifs = motif_hits(seq)

    return {
        "len": float(L), "embed": embed_vec,
        "mean_nll": float(np.sum(scalars["mean_nll"])),
        "perplexity": float(np.sum(scalars["perplexity"])),
        "entropy": float(np.sum(scalars["entropy"])),
        "margin": float(np.sum(scalars["margin"])),
        "comp20": aa_composition(seq),
        "gravy": gravy(seq),
        "net_charge": net_charge_pH7(seq),
        "aromatic_frac": aromatic_fraction(seq),
        "hydro_frac": hydro_fraction(seq),
        "max_hydrophobic_run": float(max_hydrophobic_run(seq)),
        "solubility_score": sol_sc,
        "protease_stability_score": stab_sc,
        "aggregation_score": agg_sc,
        "muH": float(muH),
        "amphipathic_idx": float(amph),
        "cys_count": int(nC),
        "cys_pairs": int(nC_pairs),
        "cys_mean_gap": float(c_gap),
        "motif_hits": int(motifs),
    }

def feat_vector(feat: Dict[str, Any], mode: str = "fuse") -> np.ndarray:
    scalars = np.array([
        feat["mean_nll"], feat["perplexity"], feat["entropy"], feat["margin"], feat["len"],
        feat["muH"], feat["amphipathic_idx"], feat["cys_count"], feat["cys_pairs"], feat["cys_mean_gap"], feat["motif_hits"]
    ], dtype=np.float32)
    return {
        "embed": feat["embed"],
        "scalars": np.concatenate([scalars, feat["comp20"]]),
        "fuse": np.concatenate([feat["embed"], feat["comp20"], scalars]),
    }[mode]

# ===================== Heads (load/save/utils) =====================
def head_path(task_name, heads_dir): return os.path.join(heads_dir, f"{task_name}.joblib")
def have_head(task_name, heads_dir): return os.path.exists(head_path(task_name, heads_dir))
def load_head(task_name, heads_dir): return joblib.load(head_path(task_name, heads_dir))
def save_head(task_name, payload, heads_dir): os.makedirs(heads_dir, exist_ok=True); joblib.dump(payload, head_path(task_name, heads_dir))

# ===================== Guards, OOD, ECE =====================
def add_guard_heuristics(out_df: pd.DataFrame) -> pd.DataFrame:
    L = out_df["length"].to_numpy(float)
    charge = out_df["net_charge_heur"].to_numpy(float)
    hydro  = out_df["hydro_frac_heur"].to_numpy(float)
    arom   = out_df["aromatic_frac_heur"].to_numpy(float)
    maxrun = out_df["max_hydro_run_heur"].to_numpy(float)

    g_len  = (L < 8) | (L > 60)
    rho    = charge / np.maximum(L, 1.0)
    g_cpp  = (rho >= 0.60) & (hydro <= 0.20) & (arom <= 0.20)
    g_hyd  = (hydro >= 0.90) & (maxrun >= 8)

    guard  = g_len | g_cpp | g_hyd
    reason = np.where(g_len, "length",
              np.where(g_cpp, "cpp_trap",
              np.where(g_hyd, "hydro_trap","")))
    out_df["GUARD_flag"] = guard
    out_df["GUARD_reason"] = reason
    out_df["charge_density"] = rho
    return out_df

def add_ood_flags(out_df: pd.DataFrame) -> pd.DataFrame:
    L = out_df["length"].to_numpy(float)
    ood = (L < 8) | (L > 60)
    out_df["OOD_flag"] = ood
    out_df["OOD_reason"] = np.where(ood, "length", "")
    return out_df

def ece_score(y_true: np.ndarray, p: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0,1,n_bins+1); ece=0.0; N=len(y_true)
    for i in range(n_bins):
        idx = (p >= bins[i]) & (p < bins[i+1])
        if np.any(idx):
            conf = np.mean(p[idx])
            acc = np.mean(y_true[idx])
            ece += (np.sum(idx) / N) * abs(acc - conf)
    return float(ece)

# ===================== Grouping =====================
def peptide_groups(seqs: List[str]) -> np.ndarray:
    def sig(s):
        if len(s) < 3: return hashlib.sha1(("*"+s).encode()).hexdigest()[:8]
        kmers = {s[i:i+3] for i in range(len(s)-2)}
        return hashlib.sha1("|".join(sorted(kmers)).encode()).hexdigest()[:8]
    return np.array([sig(s) for s in seqs])

# ===================== Thresholding & Conformal =====================
def tune_threshold_for_precision(y_true, p, target_prec=0.80) -> float:
    prec, rec, thr = precision_recall_curve(y_true, p)
    thr = np.r_[thr, 1.0]
    ok = np.where(prec >= target_prec)[0]
    return float(thr[ok[0]]) if len(ok) else 0.5

def conformal_margin(p_cal, alpha=0.10) -> float:
    m = np.abs(p_cal - 0.5)
    return float(np.quantile(m, alpha))

# ===================== Δ-MutScan helpers =====================
def single_mutants(seq: str, mode: str = "fast", budget: int = 60):
    muts=[]
    letters=list(seq)
    if mode=="none":
        return muts
    if mode=="full":
        for i,ch in enumerate(letters):
            for aa in AA20:
                if aa!=ch:
                    muts.append(("{}{}{}".format(seq[:i],aa,seq[i+1:]), i, ch, aa))
        return muts
    # fast
    for i,ch in enumerate(letters):
        cand = AA_NEIGHBORS.get(ch, [])
        for aa in cand:
            if aa!=ch: muts.append(("{}{}{}".format(seq[:i],aa,seq[i+1:]), i, ch, aa))
    if len(muts)>budget:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(muts), size=budget, replace=False)
        muts = [muts[j] for j in idx]
    return muts

def probs_for_sequences(seqs, specialists, proj_dim, blend, proj_bag, head_pipe, feat_mode):
    feats = [features_for_sequence(s, specialists, proj_dim, blend, proj_bag) for s in seqs]
    X = np.stack([feat_vector(f, feat_mode) for f in feats], axis=0)
    return head_pipe.predict_proba(X)[:,1]

# ===================== PREDICT =====================
def run_predict(args):
    specs = load_all_specialists(); print(f"Loaded {len(specs)} specialists.")

    if args.sequence:
        seqs = [args.sequence]
    elif args.input_csv:
        df_in = pd.read_csv(args.input_csv)
        if args.seq_col not in df_in.columns:
            raise RuntimeError(f"seq_col '{args.seq_col}' not in CSV.")
        seqs = df_in[args.seq_col].tolist()
    else:
        raise RuntimeError("Provide --sequence or --input_csv.")

    clean_seqs = [s.strip().upper() for s in seqs if isinstance(s, str) and re.fullmatch(rf"[{AA20}]+", s.strip().upper())]
    if not clean_seqs: raise RuntimeError("No valid AA20 sequences found.")

    feats = [features_for_sequence(s, specs, args.proj_dim, args.blend_policy, args.proj_bag) for s in clean_seqs]
    X_by_mode = {m: np.stack([feat_vector(f, m) for f in feats], axis=0) for m in ["embed", "scalars", "fuse"]}

    # Core dataframe
    out = pd.DataFrame([{k: v for k, v in f.items() if k not in ("embed","comp20")} for f in feats],
                       index=clean_seqs).reset_index().rename(columns={"index":"sequence"})
    out = out.rename(columns={
        "len":"length",
        "gravy":"gravy_heur",
        "net_charge":"net_charge_heur",
        "aromatic_frac":"aromatic_frac_heur",
        "hydro_frac":"hydro_frac_heur",
        "max_hydrophobic_run":"max_hydro_run_heur",
        "perplexity":"LM_perplexity",
        "entropy":"LM_entropy",
        "margin":"LM_margin",
        "solubility_score":"Solubility_score",
        "protease_stability_score":"ProteaseStability_score",
        "aggregation_score":"Aggregation_score",
        "muH":"muH_helix",
        "amphipathic_idx":"amphipathic_idx",
        "cys_count":"Cys_count",
        "cys_pairs":"Cys_pairs",
        "cys_mean_gap":"Cys_mean_gap",
        "motif_hits":"motif_hits",
    })

    out["AMP_proxy"] = [amp_activity_proxy_from_heuristics(s) for s in out["sequence"]]
    if len(out) > 1:
        for sn in ["Solubility_score","ProteaseStability_score","Aggregation_score"]:
            raw = out[sn].to_numpy(np.float64)
            out[sn.replace("_score","_proxy")] = (raw - raw.mean()) / (raw.std() + 1e-9)

    out = add_guard_heuristics(out)
    out = add_ood_flags(out)

    # ===== HEADS (classification + regression) =====
    if args.tier == "premium":
        for fname in sorted(glob.glob(os.path.join(args.heads_dir, "*.joblib"))):
            head_name = os.path.splitext(os.path.basename(fname))[0]
            try:
                head = joblib.load(fname)
            except Exception as e:
                print(f"⚠️ Failed loading head {head_name}: {e}")
                continue

            ttype = head.get("task_type", "classification")
            feat_mode = head.get("feat_mode", "fuse")
            X = X_by_mode.get(feat_mode, X_by_mode["fuse"])
            pipe = head["pipeline"]

            if ttype == "classification":
                metrics = head.get("metrics_cv", {})
                ece_cv = metrics.get("ECE", None)
                min_n = metrics.get("n_samples", None)
                if not args.force_unreliable_heads:
                    if (ece_cv is not None) and (ece_cv > args.max_ece):
                        print(f"⚠️ Skipping head {head_name}: ECE={ece_cv:.3f} > {args.max_ece:.2f}.")
                        continue
                    if (min_n is not None) and (min_n < max(args.min_train_samples, 30)):
                        print(f"⚠️ Skipping head {head_name}: n={min_n} too small.")
                        continue

                prob = pipe.predict_proba(X)[:,1]
                thr_auto = head.get("threshold", 0.5)
                thr = args.thr_override if args.thr_override is not None else thr_auto
                out[f"{head_name}_prob"] = prob

                if not args.no_labels:
                    if args.no_borderline:
                        dec = np.where(prob >= thr, "positive","negative")
                    else:
                        dec = np.where(np.abs(prob - thr) <= args.label_margin, "borderline",
                                       np.where(prob >= thr, "positive","negative"))
                    if args.decision_policy == "conformal":
                        qhat = head.get("conformal_margin", None)
                        if (qhat is not None) and (not args.ignore_conformal):
                            band = max(qhat * args.qhat_mult, args.label_margin)
                            undecided = np.abs(prob - thr) < band
                            dec = np.where(undecided, "abstain", dec)
                    elif args.decision_policy == "margin_only":
                        undecided = np.abs(prob - thr) < max(1e-9, args.label_margin)
                        dec = np.where(undecided, "abstain", dec)
                    if getattr(args, "abstain_on_guard", False) and not getattr(args, "ignore_guard", False):
                        dec = np.where(out["GUARD_flag"].to_numpy(bool), "abstain", dec)

                    out[f"{head_name}_label"] = (prob >= thr).astype(int)
                    out[f"{head_name}_decision"] = dec

                # ========= Δ-MutScan (robustness) =========
                if args.mutscan != "none":
                    robust_cols = ["Robust_min_drop","Robust_worst_prob","Robust_flip_count",
                                   "Robust_worst_mut","Robust_worst_pos","Robust_worst_from","Robust_worst_to"]
                    for c in robust_cols:
                        if c not in out.columns:
                            out[c] = np.nan if c!="Robust_worst_mut" else ""

                    for idx, seq in enumerate(out["sequence"].tolist()):
                        muts = single_mutants(seq, mode=args.mutscan, budget=args.mutscan_budget)
                        if not muts: continue
                        mut_seqs = [m[0] for m in muts]
                        mut_probs = probs_for_sequences(mut_seqs, specs, args.proj_dim, args.blend_policy,
                                                        args.proj_bag, pipe, feat_mode)
                        base_p = prob[idx]
                        drops = base_p - mut_probs
                        worst_j = int(np.argmax(drops))
                        out.at[idx, "Robust_min_drop"]   = float(np.max(drops))
                        out.at[idx, "Robust_worst_prob"] = float(mut_probs[worst_j])
                        out.at[idx, "Robust_flip_count"] = int(np.sum(mut_probs < thr))
                        wmut, pos, fr, to = muts[worst_j]
                        out.at[idx, "Robust_worst_mut"]  = wmut
                        out.at[idx, "Robust_worst_pos"]  = int(pos)
                        out.at[idx, "Robust_worst_from"] = fr
                        out.at[idx, "Robust_worst_to"]   = to

                        if not args.no_labels and (args.decision_policy != "none"):
                            cur = out.at[idx, f"{head_name}_decision"]
                            if cur != "abstain":
                                if (float(np.max(drops)) >= args.mutscan_abstain_drop) or (int(np.sum(mut_probs < thr)) >= 1):
                                    out.at[idx, f"{head_name}_decision"] = "abstain"

            elif ttype == "regression":
                try:
                    yhat = pipe.predict(X)
                    out[f"{head_name}_pred"] = yhat.astype(float)
                    # Safety clip (units e.g., µM) — adjust if needed:
                    out[f"{head_name}_pred"] = np.clip(out[f"{head_name}_pred"], 0.0, 1e9)
                except Exception as e:
                    print(f"⚠️ Regression head {head_name} failed: {e}")
            else:
                print(f"⏭️  Unknown head type for {head_name}: {ttype}")

    # ----- Optional: apply adapter calibration -----
    if getattr(args, "adapter_json", None):
        try:
            with open(args.adapter_json, "r") as f:
                adapter_cfg = json.load(f)
            any_prob_cols = [c for c in out.columns if c.endswith("_prob")]
            head_hint = any_prob_cols[0][:-5] if any_prob_cols else None
            out = apply_adapter_to_df(out, adapter_cfg, head_hint or "")
            print("✅ Applied adapter calibration from", args.adapter_json)
        except Exception as e:
            print(f"⚠️ Adapter load/apply failed: {e}")

    # ----- Optional: meta-head (mini-stacking) -----
    if getattr(args, "meta_joblib", None):
        try:
            meta = joblib.load(args.meta_joblib)
            feats = [s.strip() for s in (args.meta_features or "").split(",") if s.strip()]
            if not feats:
                prob_cols = [c for c in out.columns if c.endswith("_prob_cal")] or [c for c in out.columns if c.endswith("_prob")]
                base_prob = prob_cols[0] if prob_cols else None
                candidate_feats = [base_prob, "length", "net_charge_heur", "hydro_frac_heur", "AMP_proxy"]
                feats = [c for c in candidate_feats if (c is not None) and (c in out.columns)]
                print("ℹ️ meta_features (auto):", feats)
            X_meta = out[feats].astype(float).to_numpy()
            p_meta = meta.predict_proba(X_meta)[:,1]
            name = args.meta_outname
            out[f"{name}_prob"] = p_meta
            thrm = float(args.meta_threshold)
            out[f"{name}_label"] = (p_meta >= thrm).astype(int)
            dec = np.where(p_meta >= thrm, "positive", "negative")
            out[f"{name}_decision"] = dec
            print(f"✅ Applied meta-head from {args.meta_joblib} with feats={feats} thr={thrm}")
        except Exception as e:
            print(f"⚠️ Meta-head failed: {e}")

    # Save/print
    if args.out_csv:
        os.makedirs(os.path.dirname(args.out_csv) or ".", exist_ok=True)
        out.to_csv(args.out_csv, index=False)
        print(f"✅ Saved predictions to {args.out_csv}")
    else:
        print("Preview:\n", out.head(min(12, len(out))).to_string(index=False))


# ===================== TRAIN (classification only) =====================
def _parse_class_labels(y):
    y = pd.Series(y)
    try:
        yn = pd.to_numeric(y, errors="coerce")
        unique = np.unique(yn.dropna())
        if set(unique).issubset({0,1}):
            return yn.fillna(0).astype(int).to_numpy()
    except Exception:
        pass
    y = y.astype(str).str.strip().str.lower()
    return y.isin({"1", "true", "yes", "active", "pos", "positive"}).astype(int).to_numpy()

def run_train(args):
    specs = load_all_specialists(); print(f"Loaded {len(specs)} specialists.")
    df = pd.read_csv(args.dataset_csv)

    # --- Clean sequences ---
    if args.seq_col not in df.columns or args.label_col not in df.columns:
        raise RuntimeError(f"Missing columns: need '{args.seq_col}' and '{args.label_col}'.")
    df = df.dropna(subset=[args.seq_col, args.label_col]).copy()
    df[args.seq_col] = df[args.seq_col].astype(str).str.strip().str.upper()
    df = df[df[args.seq_col].str.fullmatch(rf"[{AA20}]+", na=False)].copy()
    df = df.drop_duplicates(subset=[args.seq_col]).reset_index(drop=True)

    if len(df) < args.min_train_samples:
        raise RuntimeError(f"Too few valid samples to train (found {len(df)}, need ≥{args.min_train_samples}).")

    # --- Feature extraction ---
    feats = [features_for_sequence(s, specs, args.proj_dim, args.blend_policy, args.proj_bag) for s in df[args.seq_col]]
    X = np.stack([feat_vector(f, args.feat_mode) for f in feats], axis=0)

    # --- Groups for leakage-safe CV ---
    groups = peptide_groups(df[args.seq_col].tolist())

    summary = {
        "task": args.task_name, "task_type": "classification", "feat_mode": args.feat_mode,
        "proj_dim": args.proj_dim, "blend_policy": args.blend_policy, "proj_bag": args.proj_bag,
        "n_samples": int(len(df)), "timestamp": time.time(), "precision_target": args.precision_target,
        "conformal_alpha": args.conformal_alpha, "cv_folds": args.cv_folds
    }

    y = _parse_class_labels(df[args.label_col])

    gkf = GroupKFold(n_splits=args.cv_folds)
    fold_stats, thr_list, eces = [], [], []

    base = LogisticRegression(
        penalty="l2", solver="saga", max_iter=5000, C=1.0,
        class_weight="balanced", random_state=RNG_SEED
    )

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), 1):
        # nested: split train into train_in / train_cal (80/20)
        rng = np.random.default_rng(RNG_SEED + fold)
        idx = rng.permutation(tr)
        cut = int(0.8 * len(idx))
        tr_in, tr_cal = idx[:cut], idx[cut:] if cut < len(idx) else (idx, idx)

        calibrator = "sigmoid" if args.calibration in ("platt", "sigmoid", "isotonic") else None
        if args.calibration == "isotonic" and len(tr_cal) < 200:
            calibrator = "sigmoid"
        if calibrator is not None:
            clf = CalibratedClassifierCV(base_estimator=base, method=calibrator, cv=3)
        else:
            clf = base
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
        pipe.fit(X[tr_in], y[tr_in])

        # tune threshold on tr_cal for target precision
        p_cal = pipe.predict_proba(X[tr_cal])[:,1] if len(tr_cal) > 0 else pipe.predict_proba(X[tr_in])[:,1]
        thr_fold = tune_threshold_for_precision(y[tr_cal] if len(tr_cal)>0 else y[tr_in], p_cal, args.precision_target)
        thr_list.append(thr_fold)
        qhat = conformal_margin(p_cal, args.conformal_alpha)

        p_te = pipe.predict_proba(X[te])[:,1]
        f1 = f1_score(y[te], (p_te >= thr_fold).astype(int))
        auroc = roc_auc_score(y[te], p_te)
        auprc = average_precision_score(y[te], p_te)
        brier = brier_score_loss(y[te], p_te)
        ece = ece_score(y[te], p_te, n_bins=10)
        eces.append(ece)

        fold_stats.append({
            "fold": fold, "AUROC": float(auroc), "AUPRC": float(auprc), "F1@thr": float(f1),
            "Brier": float(brier), "best_thr_prec_target": float(thr_fold), "ECE": float(ece),
            "qhat": float(qhat), "n_tr_in": int(len(tr_in)), "n_tr_cal": int(len(tr_cal)), "n_te": int(len(te))
        })

    metrics = {
        "task": args.task_name, "type": "classification", "feat_set": args.feat_mode,
        "calibration": args.calibration,
        "AUROC": float(np.mean([m["AUROC"] for m in fold_stats])),
        "AUPRC": float(np.mean([m["AUPRC"] for m in fold_stats])),
        "F1@thr": float(np.mean([m["F1@thr"] for m in fold_stats])),
        "Brier": float(np.mean([m["Brier"] for m in fold_stats])),
        "best_thr_prec_target": float(np.mean(thr_list)),
        "ECE": float(np.mean(eces)),
        "folds": fold_stats, "n_samples": int(len(df))
    }
    print("CV metrics:", json.dumps(metrics, indent=2))

    # full fit + calibration
    calibrator = "sigmoid" if args.calibration in ("platt", "sigmoid", "isotonic") else None
    if args.calibration == "isotonic" and len(df) < 500:
        calibrator = "sigmoid"
    if calibrator is not None:
        clf_full = CalibratedClassifierCV(base_estimator=base, method=calibrator, cv=5)
    else:
        clf_full = base
    pipe_full = Pipeline([("scaler", StandardScaler()), ("clf", clf_full)]).fit(X, y)
    p_full = pipe_full.predict_proba(X)[:,1]
    thr_final = tune_threshold_for_precision(y, p_full, args.precision_target)
    qhat_final = conformal_margin(p_full, args.conformal_alpha)

    payload = {
        "task_type": "classification",
        "pipeline": pipe_full,
        "feat_mode": args.feat_mode,
        "threshold": float(thr_final),
        "conformal_margin": float(qhat_final),
        "metrics_cv": metrics
    }
    save_head(args.task_name, payload, args.heads_dir)

    with open(os.path.join(args.heads_dir, f"{args.task_name}.summary.json"), "w") as f:
        json.dump({"task": args.task_name, **metrics, "threshold": float(thr_final), "qhat": float(qhat_final)}, f, indent=2)

    print(f"✅ Saved head → {head_path(args.task_name, args.heads_dir)}")


# ===================== TRAIN (regression; e.g., MIC) =====================
def run_train_regression(args):
    specs = load_all_specialists(); print(f"Loaded {len(specs)} specialists.")
    df = pd.read_csv(args.dataset_csv)
    if args.seq_col not in df.columns or args.target_col not in df.columns:
        raise RuntimeError(f"Missing columns: need '{args.seq_col}' and '{args.target_col}'.")

    # Clean
    df = df.dropna(subset=[args.seq_col, args.target_col]).copy()
    df[args.seq_col] = df[args.seq_col].astype(str).str.strip().str.upper()
    df = df[df[args.seq_col].str.fullmatch(rf"[{AA20}]+", na=False)].copy()
    df = df.drop_duplicates(subset=[args.seq_col]).reset_index(drop=True)

    # Features
    feats = [features_for_sequence(s, specs, args.proj_dim, args.blend_policy, args.proj_bag) for s in df[args.seq_col]]
    X = np.stack([feat_vector(f, args.feat_mode) for f in feats], axis=0)
    y = pd.to_numeric(df[args.target_col], errors="coerce").to_numpy(float)

    groups = peptide_groups(df[args.seq_col].tolist())
    gkf = GroupKFold(n_splits=args.cv_folds)

    maes, r2s = [], []
    for tr, te in gkf.split(X, y, groups):
        pipe = Pipeline([("s", StandardScaler()), ("m", HuberRegressor())])
        pipe.fit(X[tr], y[tr])
        pred = pipe.predict(X[te])
        maes.append(mean_absolute_error(y[te], pred))
        r2s.append(r2_score(y[te], pred))
    metrics = {"MAE_cv": float(np.mean(maes)), "R2_cv": float(np.mean(r2s)), "n_samples": int(len(df))}
    print("CV metrics (regression):", json.dumps(metrics, indent=2))

    pipe_full = Pipeline([("s", StandardScaler()), ("m", HuberRegressor())]).fit(X, y)
    payload = {"task_type": "regression", "pipeline": pipe_full, "feat_mode": args.feat_mode, "metrics_cv": metrics}
    save_head(args.task_name, payload, args.heads_dir)
    with open(os.path.join(args.heads_dir, f"{args.task_name}.summary.json"), "w") as f:
        json.dump({"task": args.task_name, **metrics, "timestamp": time.time()}, f, indent=2)
    print(f"✅ Saved regression head → {head_path(args.task_name, args.heads_dir)}")


# ===================== main =====================
def main():
    ap = argparse.ArgumentParser(description="EvoTensor Peptide Product — AMP + MIC (classification + regression) with Δ-MutScan")
    sub = ap.add_subparsers(dest="cmd", required=True)

    # Predict CLI
    p_pred = sub.add_parser("predict", help="Predict on a sequence or CSV")
    p_pred.add_argument("--tier", default=os.environ.get("TIER", "premium"), choices=["free", "premium"])
    p_pred.add_argument("--input_csv", help="CSV with sequences")
    p_pred.add_argument("--sequence", help="Single peptide (AA20)")
    p_pred.add_argument("--seq_col", default="sequence")
    p_pred.add_argument("--out_csv")
    p_pred.add_argument("--heads_dir", default=HEADS_DIR)
    p_pred.add_argument("--proj_dim", type=int, default=PROJ_DIM)
    p_pred.add_argument("--blend_policy", default=BLEND_POLICY, choices=["triangular", "inverse_dist", "hard"])
    p_pred.add_argument("--proj_bag", type=int, default=PROJ_BAG)
    # label/threshold controls
    p_pred.add_argument("--no_labels", action="store_true",
                        help="Only output probabilities; do not emit *_label/decision.")
    p_pred.add_argument("--thr_override", type=float, default=None,
                        help="Override classification threshold in [0,1]. If unset, uses tuned threshold.")
    p_pred.add_argument("--label_margin", type=float, default=0.15,
                        help="±band γύρω από το threshold που χαρακτηρίζεται 'borderline'.")
    # reliability-gating controls
    p_pred.add_argument("--force_unreliable_heads", action="store_true",
                        help="Bypass ECE/n gating and run heads anyway (for benchmarking).")
    p_pred.add_argument("--max_ece", type=float, default=0.15,
                        help="ECE threshold for reliability gate.")
    p_pred.add_argument("--min_train_samples", type=int, default=MIN_TRAIN_SAMPLES,
                        help="Minimum samples required to accept a head (gating).")
    # decision policy
    p_pred.add_argument("--decision_policy", default="conformal",
                        choices=["conformal","margin_only","none"],
                        help="Πολιτική abstention: conformal | margin_only | none.")
    p_pred.add_argument("--qhat_mult", type=float, default=1.0,
                        help="Multiplier στο conformal qhat πριν συγκριθεί με το label_margin.")
    p_pred.add_argument("--no_borderline", action="store_true",
                        help="Απενεργοποίηση ετικέτας 'borderline' (μένει μόνο positive/negative/abstain).")
    # Guard/Conformal ignores
    p_pred.add_argument("--ignore_conformal", action="store_true",
                        help="Απενεργοποιεί abstention από conformal margin (χρήσιμο για dry-runs).")
    p_pred.add_argument("--ignore_guard", action="store_true",
                        help="Απενεργοποιεί abstention από guard rules (length/CPP/hydrophobic).")
    # Δ-MutScan
    p_pred.add_argument("--mutscan", default="fast", choices=["none","fast","full"],
                        help="Robustness scan over single-aa mutants.")
    p_pred.add_argument("--mutscan_budget", type=int, default=60,
                        help="Max mutants per sequence for fast scan.")
    p_pred.add_argument("--mutscan_abstain_drop", type=float, default=0.20,
                        help="Abstain αν worst drop≥this ή υπάρχει οποιοδήποτε flip κάτω από threshold.")
    p_pred.add_argument("--abstain_on_guard", action="store_true",
                        help="Αν είναι on, κάνε abstain όταν ενεργοποιούνται guard rules (length/CPP/hydrophobic).")
    # --- Optional adapters / meta-head (post-process) ---
    p_pred.add_argument("--adapter_json", type=str, default=None,
                        help="Μικρό Platt adapter JSON με κλειδιά a,b[,head,thr]. Εφαρμόζεται στα *_prob.")
    p_pred.add_argument("--meta_joblib", type=str, default=None,
                        help="Joblib αρχείο ενός ελαφρού meta-classifier που δέχεται features από το output.")
    p_pred.add_argument("--meta_features", type=str, default="",
                        help="Comma-separated λίστα στηλών (features) από το output CSV για το meta-head.")
    p_pred.add_argument("--meta_outname", type=str, default="meta",
                        help="Prefix για τα output columns του meta-head (π.χ. 'meta').")
    p_pred.add_argument("--meta_threshold", type=float, default=0.5,
                        help="Threshold για το meta-head.")
    p_pred.set_defaults(func=run_predict)

    # Train CLI (CLASSIFICATION ONLY)
    p_train = sub.add_parser("train", help="Train a classification head from CSV")
    p_train.add_argument("--task_name", required=True)
    p_train.add_argument("--dataset_csv", required=True)
    p_train.add_argument("--seq_col", default="sequence")
    p_train.add_argument("--label_col", required=True)
    p_train.add_argument("--feat_mode", default="fuse", choices=["embed", "scalars", "fuse"])
    p_train.add_argument("--calibration", default="sigmoid", choices=["none", "platt", "sigmoid", "isotonic"])
    p_train.add_argument("--heads_dir", default=HEADS_DIR)
    p_train.add_argument("--proj_dim", type=int, default=PROJ_DIM)
    p_train.add_argument("--blend_policy", default=BLEND_POLICY, choices=["triangular", "inverse_dist", "hard"])
    p_train.add_argument("--proj_bag", type=int, default=PROJ_BAG)
    p_train.add_argument("--min_train_samples", type=int, default=MIN_TRAIN_SAMPLES)
    p_train.add_argument("--precision_target", type=float, default=PRECISION_TARGET)
    p_train.add_argument("--conformal_alpha", type=float, default=CONFORMAL_ALPHA)
    p_train.add_argument("--cv_folds", type=int, default=CV_FOLDS)
    p_train.set_defaults(func=run_train)

    # Train CLI (REGRESSION: e.g., MIC)
    p_train_r = sub.add_parser("train_reg", help="Train a regression head (e.g., MIC) from CSV")
    p_train_r.add_argument("--task_name", required=True)
    p_train_r.add_argument("--dataset_csv", required=True)
    p_train_r.add_argument("--seq_col", default="sequence")
    p_train_r.add_argument("--target_col", required=True, help="Numeric target column (e.g., MIC_uM)")
    p_train_r.add_argument("--feat_mode", default="fuse", choices=["embed", "scalars", "fuse"])
    p_train_r.add_argument("--heads_dir", default=HEADS_DIR)
    p_train_r.add_argument("--proj_dim", type=int, default=PROJ_DIM)
    p_train_r.add_argument("--blend_policy", default=BLEND_POLICY, choices=["triangular", "inverse_dist", "hard"])
    p_train_r.add_argument("--proj_bag", type=int, default=PROJ_BAG)
    p_train_r.add_argument("--cv_folds", type=int, default=CV_FOLDS)
    p_train_r.set_defaults(func=run_train_regression)

    args = ap.parse_args()
    args.func(args)

if __name__ == "__main__":
    main()


Writing /kaggle/working/pep_product.py


In [ ]:
!python pep_product.py predict \
  --tier premium \
  --input_csv /kaggle/working/ultimate_test_data_level_10/ultimate_train_10.csv \
  --heads_dir /kaggle/working/heads \
  --adapter_json /kaggle/working/adapter.json \
  --out_csv preds_cal.csv


# simulation

In [5]:
%%writefile labsim_demo.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
labsim_demo.py — Lightweight PK/PD "disease plugin" simulator for peptides.

Inputs:
  - peptides CSV (at least: sequence; ideally AMP_activity_*_prob or AMP_proxy,
    and optional MIC_*_pred, ProteaseStability_* , Aggregation_*, gravy_heur)
  - built-in plugin key (mrsa_demo, cancer_invitro_demo) or JSON path with params
  - dosing schedule (dose_uM / interval_h / n_doses / duration_h)

Outputs:
  - results CSV sorted by efficacy (with simple safety penalty)
  - per-peptide summary (kill@24h or viability@72h, suggested regimen)

This is intentionally simple (proxy-based) to support demos & what-if.
"""

import argparse, json, math, sys, os
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

# ----------------------- Built-in disease plugins ----------------------------

def builtin_plugin(key: str) -> Dict[str, Any]:
    key = (key or "").lower().strip()
    if key == "mrsa_demo":
        return {
            "name": "MRSA (demo)",
            "type": "bacteria",
            "readout": "CFU_over_time",
            "growth_model": "logistic",
            "params": {"r_per_h": 0.08, "K": 1e9, "N0": 1e6},
            "pk": {"model": "one_compartment", "kel_per_h": 0.04},
            "pd": {
                # how to map peptide features -> EC50, kmax, hill
                "ec50": {"from": ["MIC_ultimate_pred","MIC_static_pred","MIC_pred_uM","AMP_proxy","AMP_activity_prob"]},
                "kmax": {"base": 0.03, "scale_prob": 0.04},   # per hour
                "hill":  {"base": 1.3}
            },
            "safety": {
                "tox_from": ["Aggregation_*","gravy_heur"],
                "alpha": 0.2, "beta": 1.0
            },
            "endpoint_hours": 24
        }
    if key == "cancer_invitro_demo":
        return {
            "name": "Cancer in vitro (demo)",
            "type": "cancer",
            "readout": "Viability_vs_time",
            "params": {"Viab0": 1.0},
            "pk": {"model": "one_compartment", "kel_per_h": 0.03},
            "pd": {
                "ec50": {"from": ["IC50_uM","MIC_ultimate_pred","AMP_proxy","AMP_activity_prob"]},
                "emax": {"base": 0.9},   # max fractional effect
                "hill": {"base": 1.4}
            },
            "safety": {
                "tox_from": ["Aggregation_*","gravy_heur"],
                "alpha": 0.25, "beta": 1.0
            },
            "endpoint_hours": 72
        }
    raise ValueError(f"Unknown builtin plugin: {key}")


# ----------------------- Utilities ----------------------------

def first_existing(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c.endswith("*"):
            # wildcard prefix
            prefix = c[:-1]
            hits = [col for col in df.columns if col.startswith(prefix)]
            if hits:
                return hits[0]
        else:
            if c in df.columns:
                return c
    return None

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0/(1.0 + np.exp(-x))

def soft_clip(x, lo, hi):
    return max(lo, min(hi, x))

def ensure_cols(df: pd.DataFrame, cols: List[str]) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

# ----------------------- PK/PD Mappings (proxy) ----------------------------

def map_ec50_uM(row: pd.Series, pd_cfg: Dict[str, Any]) -> float:
    """
    Heuristic mapping from available columns -> EC50 in μM (lower is better).
    Priority:
      - If MIC/IC50-like column exists, derive EC50 from it.
      - Else from AMP_proxy / AMP_activity_prob (prob in 0..1).
    Returns a clipped EC50 in [0.25, 256] μM.
    """
    ec50_sources = pd_cfg.get("ec50", {}).get("from", [])
    col = first_existing(row.to_frame().T, ec50_sources)
    if col:
        val = float(row[col])
        # If it's obviously a probability, convert; else assume μM-like
        if 0.0 <= val <= 1.0:
            # map prob → EC50 exponentially: prob=1 → ~1 μM, prob=0 → ~128 μM
            ec50 = 1.0 * (128.0 ** (1.0 - val))
        else:
            # if MIC predicted, EC50 ≈ MIC/4 (toy)
            ec50 = max(0.25, float(val)/4.0)
    else:
        # fallback on generic prob-like signal
        p = 0.5
        for k in ["AMP_activity_prob","AMP_activity_ultimate_prob","AMP_proxy","prob"]:
            if k in row and pd.notna(row[k]):
                p = float(row[k]); break
        ec50 = 1.0 * (128.0 ** (1.0 - soft_clip(p,0.0,1.0)))
    return float(np.clip(ec50, 0.25, 256.0))

def map_kmax_per_h(row: pd.Series, pd_cfg: Dict[str, Any]) -> float:
    base = float(pd_cfg.get("kmax", {}).get("base", 0.03))
    scale = float(pd_cfg.get("kmax", {}).get("scale_prob", 0.04))
    p = 0.5
    for k in ["AMP_activity_prob","AMP_activity_ultimate_prob","AMP_proxy","prob"]:
        if k in row and pd.notna(row[k]):
            p = float(row[k]); break
    return base + scale * soft_clip(p,0.0,1.0)

def map_hill(row: pd.Series, pd_cfg: Dict[str, Any]) -> float:
    return float(pd_cfg.get("hill", {}).get("base", 1.3))

def map_emax(row: pd.Series, pd_cfg: Dict[str, Any]) -> float:
    return float(pd_cfg.get("emax", {}).get("base", 0.9))

def map_kel_per_h(row: pd.Series, plugin: Dict[str, Any]) -> float:
    kel = float(plugin.get("pk", {}).get("kel_per_h", 0.04))
    # crude tweak: better protease stability ⇒ slower elimination
    for k in ["ProteaseStability_score","ProteaseStability_proxy"]:
        if k in row and pd.notna(row[k]):
            s = float(row[k])
            kel *= (1.0 - 0.15*np.tanh(s))  # ±15%
            break
    return max(0.002, min(1.0, kel))

def tox_penalty(row: pd.Series, safety_cfg: Dict[str, Any], C: float) -> float:
    """
    Produce a small penalty [0..0.5] increasing with aggregation/hydrophobicity and dose.
    """
    alpha = float(safety_cfg.get("alpha", 0.2))
    beta  = float(safety_cfg.get("beta", 1.0))
    sig = 0.0
    for key in safety_cfg.get("tox_from", []):
        if key.endswith("*"):
            prefix = key[:-1]
            cand = [c for c in row.index if c.startswith(prefix)]
            if cand:
                v = float(row[cand[0]])
                sig += sigmoid(beta * v)
        else:
            if key in row and pd.notna(row[key]):
                v = float(row[key]); sig += sigmoid(beta * v)
    # slightly scale with concentration (more exposure → more risk)
    return float(np.clip(alpha * sig * (1.0 + 0.01*C), 0.0, 0.5))

# ----------------------- Schedules & Simulation ----------------------------

@dataclass
class Dosing:
    dose_uM: float
    interval_h: float
    n_doses: int

def schedule_times(d: Dosing) -> List[float]:
    return [i*d.interval_h for i in range(d.n_doses)]

def simulate_bacteria(plugin: Dict[str, Any], row: pd.Series, dosing: Dosing, duration_h: float, dt_h: float=0.25) -> Dict[str, Any]:
    params = plugin["params"]
    r = float(params["r_per_h"]); K = float(params["K"]); N = float(params["N0"])
    # Map PD & PK
    pd_cfg = plugin["pd"]
    EC50 = map_ec50_uM(row, pd_cfg)
    kmax = map_kmax_per_h(row, pd_cfg)
    h    = map_hill(row, pd_cfg)
    kel  = map_kel_per_h(row, plugin)
    # simulate
    tgrid = np.arange(0.0, duration_h+1e-9, dt_h)
    C = 0.0
    times = set(schedule_times(dosing))
    for t in tgrid:
        if any(abs(t - ti) < 1e-9 for ti in times):
            C += dosing.dose_uM  # bolus
        # kill
        kk = kmax * (C**h)/(EC50**h + C**h)
        dN = r*N*(1.0 - N/K)*dt_h - kk*N*dt_h
        N = max(1.0, N + dN)
        # PK decay
        C = max(0.0, C * math.exp(-kel*dt_h))
    # endpoint
    N0 = float(params["N0"]); kill_log10 = math.log10(N0) - math.log10(N)
    # crude safety penalty at peak C (approx)
    peakC = dosing.dose_uM  # after first dose
    penalty = tox_penalty(row, plugin.get("safety",{}), peakC)
    score = kill_log10 - penalty
    return {
        "EC50_uM": EC50, "kmax_per_h": kmax, "hill": h, "kel_per_h": kel,
        "CFU_start": N0, "CFU_end": N, "kill_log10": kill_log10,
        "safety_penalty": penalty, "efficacy_score": score
    }

def simulate_cancer(plugin: Dict[str, Any], row: pd.Series, dosing: Dosing, duration_h: float, dt_h: float=0.25) -> Dict[str, Any]:
    params = plugin["params"]
    Viab = float(params.get("Viab0", 1.0))
    pd_cfg = plugin["pd"]
    EC50 = map_ec50_uM(row, pd_cfg)
    Emax = map_emax(row, pd_cfg)
    h    = map_hill(row, pd_cfg)
    kel  = map_kel_per_h(row, plugin)
    # PK
    tgrid = np.arange(0.0, duration_h+1e-9, dt_h)
    C = 0.0
    times = set(schedule_times(dosing))
    for t in tgrid:
        if any(abs(t - ti) < 1e-9 for ti in times):
            C += dosing.dose_uM
        effect = Emax * (C**h)/(EC50**h + C**h)
        # Update viability as a smooth decay toward (1 - effect)
        target = 1.0 - effect
        Viab += (target - Viab) * 0.25  # relax toward target
        Viab = float(np.clip(Viab, 0.0, 1.5))
        C = max(0.0, C * math.exp(-kel*dt_h))
    peakC = dosing.dose_uM
    penalty = tox_penalty(row, plugin.get("safety",{}), peakC)
    # lower Viab is better; invert to an efficacy score
    score = (1.0 - Viab) - penalty
    return {
        "EC50_uM": EC50, "Emax": Emax, "hill": h, "kel_per_h": kel,
        "Viability_end": Viab, "kill_fraction": 1.0 - Viab,
        "safety_penalty": penalty, "efficacy_score": score
    }

# ----------------------- CLI ----------------------------

def main():
    ap = argparse.ArgumentParser(description="Peptide PK/PD demo simulator (disease plugins).")
    ap.add_argument("--peptides_csv", required=True, help="CSV with at least 'sequence'. Include AMP_activity_prob/AMP_proxy/MIC_pred if available.")
    ap.add_argument("--plugin", required=False, default="mrsa_demo", help="Builtin key (mrsa_demo,cancer_invitro_demo) or JSON path.")
    ap.add_argument("--dose_uM", type=float, default=32.0)
    ap.add_argument("--interval_h", type=float, default=12.0)
    ap.add_argument("--n_doses", type=int, default=2)
    ap.add_argument("--duration_h", type=float, default=24.0)
    ap.add_argument("--out_csv", required=True)
    args = ap.parse_args()

    # Load peptides
    df = pd.read_csv(args.peptides_csv)
    if "sequence" not in df.columns:
        raise ValueError("peptides_csv must contain a 'sequence' column.")
    df = df.copy()

    # Load plugin
    if os.path.isfile(args.plugin):
        with open(args.plugin, "r") as f:
            plugin = json.load(f)
    else:
        plugin = builtin_plugin(args.plugin)

    # If cancer demo and duration not set, set to plugin default
    if "endpoint_hours" in plugin and (args.duration_h is None or args.duration_h <= 0):
        args.duration_h = float(plugin["endpoint_hours"])

    dosing = Dosing(dose_uM=args.dose_uM, interval_h=args.interval_h, n_doses=args.n_doses)

    rows = []
    for i, row in df.iterrows():
        base = {"sequence": row["sequence"]}
        try:
            if plugin["type"] == "bacteria":
                res = simulate_bacteria(plugin, row, dosing, duration_h=args.duration_h)
                base.update({
                    "Endpoint_h": plugin.get("endpoint_hours", args.duration_h),
                    "kill_log10": res["kill_log10"],
                    "CFU_end": res["CFU_end"],
                    "safety_penalty": res["safety_penalty"],
                    "efficacy_score": res["efficacy_score"],
                    "EC50_uM": res["EC50_uM"], "kmax_per_h": res["kmax_per_h"], "hill": res["hill"], "kel_per_h": res["kel_per_h"]
                })
            elif plugin["type"] == "cancer":
                res = simulate_cancer(plugin, row, dosing, duration_h=args.duration_h)
                base.update({
                    "Endpoint_h": plugin.get("endpoint_hours", args.duration_h),
                    "Viability_end": res["Viability_end"],
                    "kill_fraction": res["kill_fraction"],
                    "safety_penalty": res["safety_penalty"],
                    "efficacy_score": res["efficacy_score"],
                    "EC50_uM": res["EC50_uM"], "Emax": res["Emax"], "hill": res["hill"], "kel_per_h": res["kel_per_h"]
                })
            else:
                raise ValueError(f"Unsupported plugin type: {plugin['type']}")
        except Exception as e:
            base.update({"error": str(e)})
        rows.append(base)

    out = pd.DataFrame(rows)
    # Rank by efficacy_score (desc)
    if "efficacy_score" in out.columns:
        out = out.sort_values("efficacy_score", ascending=False)

    out.to_csv(args.out_csv, index=False)
    print(f"Saved simulation results → {args.out_csv}")
    # quick top-5 preview
    print(out.head(5).to_string(index=False))


if __name__ == "__main__":
    main()


Writing labsim_demo.py


!python labsim_demo.py \
  --peptides_csv /kaggle/working/preds_none.csv \
  --plugin mrsa_demo \
  --dose_uM 32 \
  --interval_h 12 \
  --n_doses 2 \
  --duration_h 24 \
  --out_csv /kaggle/working/mrsa_sim.csv
